## Dataset preparation using principal dataset, orf subset. creating a dataset of 300 randomly selected images.


Preparing JUMP ORF Dataset for μSplit
In this notebook we will do the following:
1. Load the ORF subset from the principal JUMP dataset
2. Retrieve gene-level metadata using broad-babel
3. Sample 300 images randomly from available ORF perturbations
4. Download and combine channels for μSplit training
5. Save organized dataset with comprehensive metadata

In [1]:
import os
import itertools
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.colors as mpl
import polars as pl
import requests
import json
from jump_portrait.fetch import get_item_location_info, get_jump_image
from broad_babel.query import get_mapper

## Load and explore the data

In [2]:
# Load the ORF dataset using the principal dataset manifest approach
print("=== Loading JUMP ORF Dataset Manifest ===")
INDEX_FILE = "https://raw.githubusercontent.com/jump-cellpainting/datasets/v0.11.0/manifests/profile_index.json"

response = requests.get(INDEX_FILE)
response.raise_for_status()
profile_index = response.json()

# Find ORF subset URL
orf_url = None
for dataset in profile_index:
    if dataset['subset'] == 'orf':
        orf_url = dataset['url']
        break
        
if not orf_url:
    raise ValueError("Could not find 'orf' subset in the manifest.")

print(f"ORF dataset URL: {orf_url}")

print("\n=== Scanning ORF Parquet file ===")
profiles = pl.scan_parquet(orf_url)
print("Scan complete.")

=== Loading JUMP ORF Dataset Manifest ===
ORF dataset URL: https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump-assembled/source_all/workspace/profiles_assembled/ORF/v1.0a/profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony.parquet

=== Scanning ORF Parquet file ===
Scan complete.


In [3]:
print("\n=== JUMP ORF Profile Data ===")
meta_cols = profiles.select(pl.col("^Metadata.*$")).collect_schema().names()
profile_rows = profiles.select(pl.len()).collect().item()
profile_cols = profiles.collect_schema().len()

print(f"Number of profiles: {profile_rows}")
print(f"Number of features: {profile_cols}")
print(f"Number of metadata columns: {len(meta_cols)}")
print(f"Metadata columns: {meta_cols}")


=== JUMP ORF Profile Data ===
Number of profiles: 81660
Number of features: 726
Number of metadata columns: 4
Metadata columns: ['Metadata_Source', 'Metadata_Plate', 'Metadata_Well', 'Metadata_JCP2022']


In [4]:
# ORF Dataset Visualization - Similar to Pilot Dataset Exploration
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def visualize_orf_dataset_overview(profiles):
    """Create comprehensive overview visualizations of the ORF dataset"""
    
    # Collect basic data
    collected_profiles = profiles.collect()
    total_profiles = len(collected_profiles)
    unique_sources = collected_profiles['Metadata_Source'].unique()
    unique_plates = collected_profiles['Metadata_Plate'].unique()
    unique_orfs = collected_profiles['Metadata_JCP2022'].unique()
    
    # Count profiles per ORF
    orf_counts = Counter(collected_profiles['Metadata_JCP2022'].to_list())
    profile_distribution = Counter(orf_counts.values())
    
    # Create comprehensive figure
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Dataset Overview (top section)
    gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)
    
    # Dataset statistics
    ax_stats = fig.add_subplot(gs[0, :2])
    ax_stats.axis('off')
    
    stats_text = f"""
    JUMP ORF Dataset Overview
    ════════════════════════
    
    Total Profiles:     {total_profiles:,}
    Unique ORFs:        {len(unique_orfs):,}
    Data Sources:       {len(unique_sources)}
    Unique Plates:      {len(unique_plates):,}
    Feature Columns:    {len(collected_profiles.columns) - 4:,}
    Metadata Columns:   4
    
    Profile Distribution per ORF:
    • {profile_distribution[5]:,} ORFs ({profile_distribution[5]/len(unique_orfs)*100:.1f}%) have 5 profiles
    • {sum(v for k, v in profile_distribution.items() if k != 5):,} ORFs have other counts
    """
    
    ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes, 
                  fontsize=12, verticalalignment='top', fontfamily='monospace')
    
    # 2. Profile count distribution histogram
    ax_hist = fig.add_subplot(gs[0, 2:])
    counts = list(orf_counts.values())
    bins = np.arange(1, max(counts) + 2) - 0.5
    ax_hist.hist(counts, bins=bins, alpha=0.7, color='steelblue', edgecolor='black')
    ax_hist.set_xlabel('Number of Profiles per ORF')
    ax_hist.set_ylabel('Number of ORFs')
    ax_hist.set_title('Distribution of Profile Counts per ORF')
    ax_hist.grid(True, alpha=0.3)
    
    # Add text annotations for key statistics
    ax_hist.axvline(5, color='red', linestyle='--', alpha=0.8)
    ax_hist.text(5.2, ax_hist.get_ylim()[1]*0.8, f'Modal: 5 profiles\n({profile_distribution[5]:,} ORFs)', 
                 fontsize=10, color='red')
    
    # 3. Source and Plate distribution
    ax_sources = fig.add_subplot(gs[1, 0])
    source_counts = Counter(collected_profiles['Metadata_Source'].to_list())
    ax_sources.bar(range(len(source_counts)), list(source_counts.values()), color='lightcoral')
    ax_sources.set_xlabel('Data Source')
    ax_sources.set_ylabel('Number of Profiles')
    ax_sources.set_title('Profiles by Source')
    ax_sources.set_xticks(range(len(source_counts)))
    ax_sources.set_xticklabels(list(source_counts.keys()), rotation=45)
    
    # 4. Feature value distribution sample
    ax_features = fig.add_subplot(gs[1, 1:])
    # Sample some feature columns for distribution
    feature_cols = [col for col in collected_profiles.columns if col.startswith('X_')]
    sample_features = np.random.choice(feature_cols, min(5, len(feature_cols)), replace=False)
    
    for i, col in enumerate(sample_features):
        values = collected_profiles[col].to_numpy()
        ax_features.hist(values, bins=50, alpha=0.6, label=f'{col}', density=True)
    
    ax_features.set_xlabel('Feature Values')
    ax_features.set_ylabel('Density')
    ax_features.set_title('Sample Feature Distributions (Normalized)')
    ax_features.legend()
    ax_features.grid(True, alpha=0.3)
    
    return fig

def visualize_orf_sample_images(selected_orfs, num_samples=3, channels=["DNA", "RNA", "ER", "AGP", "Mito"]):
    """Visualize sample ORF images with all channels - similar to pilot exploration"""
    
    # Channel color mapping (similar to pilot)
    channel_colors = {
        "DNA": "blue",      # Hoechst - blue
        "RNA": "yellow",    # SYTO 14 - yellow 
        "ER": "green",      # Concanavalin A - green
        "AGP": "orange",    # Phalloidin/WGA - orange
        "Mito": "red"       # MitoTracker - red
    }
    
    fig, axes = plt.subplots(num_samples, len(channels) + 1, figsize=(24, 6*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    successfully_processed = 0
    
    for sample_idx in range(num_samples):
        if sample_idx >= len(selected_orfs):
            break
            
        orf_id = selected_orfs[sample_idx]
        print(f"Processing sample {sample_idx + 1}: {orf_id}")
        
        try:
            # Get image location info
            orf_info = get_item_location_info(orf_id)
            
            if orf_info.shape[0] == 0:
                print(f"  No images found for {orf_id}")
                continue
            
            # Get first available image
            test_row = orf_info.row(0)
            source = test_row[orf_info.columns.index("Metadata_Source")]
            batch = test_row[orf_info.columns.index("Metadata_Batch")]
            plate = test_row[orf_info.columns.index("Metadata_Plate")]
            well = test_row[orf_info.columns.index("Metadata_Well")]
            site = test_row[orf_info.columns.index("Metadata_Site")]
            
            print(f"  Location: {source}/{batch}/{plate}/{well}/site_{site}")
            
            # Load all channel images
            channel_images = {}
            all_imgs_norm = []
            
            for ch_idx, channel in enumerate(channels):
                try:
                    img = get_jump_image(source, batch, plate, well, channel, site, None)
                    channel_images[channel] = img
                    
                    # Normalize for display
                    img_norm = img.astype(float)
                    if img_norm.max() > img_norm.min():
                        img_norm = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min())
                    all_imgs_norm.append(img_norm)
                    
                    # Create custom colormap for this channel
                    color = channel_colors.get(channel, "gray")
                    cmap = mcolors.LinearSegmentedColormap.from_list(
                        f"custom_{color}", ["black", color]
                    )
                    
                    # Display individual channel
                    axes[sample_idx, ch_idx].imshow(img_norm, cmap=cmap)
                    axes[sample_idx, ch_idx].set_title(f"{channel}\n({img.shape})")
                    axes[sample_idx, ch_idx].axis('off')
                    
                except Exception as e:
                    print(f"  Error loading {channel}: {str(e)}")
                    axes[sample_idx, ch_idx].text(0.5, 0.5, f"Error\n{channel}", 
                                                ha='center', va='center', 
                                                transform=axes[sample_idx, ch_idx].transAxes)
                    axes[sample_idx, ch_idx].axis('off')
            
            # Create composite image (if we have all channels)
            if len(all_imgs_norm) == 5:
                composite = np.zeros((*all_imgs_norm[0].shape, 3))
                
                # Combine channels with appropriate color weighting
                # DNA (blue channel)
                composite[..., 2] = all_imgs_norm[0]
                # ER (green) + RNA (yellow-green contribution)  
                composite[..., 1] = 0.7*all_imgs_norm[2] + 0.3*all_imgs_norm[1]
                # Mito (red) + AGP (orange-red contribution)
                composite[..., 0] = 0.7*all_imgs_norm[4] + 0.3*all_imgs_norm[3]
                
                axes[sample_idx, -1].imshow(composite)
                axes[sample_idx, -1].set_title("5-Channel\nComposite")
                axes[sample_idx, -1].axis('off')
            
            # Add ORF label on the left
            axes[sample_idx, 0].text(-0.1, 0.5, f"ORF:\n{orf_id}", 
                                   rotation=90, ha='center', va='center', 
                                   transform=axes[sample_idx, 0].transAxes, fontsize=10)
            
            successfully_processed += 1
            
        except Exception as e:
            print(f"  Failed to process {orf_id}: {str(e)}")
            for col_idx in range(len(channels) + 1):
                axes[sample_idx, col_idx].text(0.5, 0.5, f"Error\n{orf_id}", 
                                             ha='center', va='center',
                                             transform=axes[sample_idx, col_idx].transAxes)
                axes[sample_idx, col_idx].axis('off')
    
    plt.suptitle(f'ORF Sample Images - {successfully_processed}/{num_samples} Successfully Loaded', 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    
    return fig

def visualize_orf_metadata_distributions(profiles, selected_orfs):
    """Create detailed metadata distribution plots"""
    
    collected_profiles = profiles.collect()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    # 1. Well distribution
    wells = collected_profiles['Metadata_Well'].to_list()
    well_counts = Counter(wells)
    
    # Create well plate heatmap
    ax = axes[0]
    well_matrix = np.zeros((16, 24))  # Standard 384-well plate
    
    for well, count in well_counts.items():
        if len(well) >= 3:
            row = ord(well[0]) - ord('A')  # A=0, B=1, etc.
            col = int(well[1:]) - 1  # 01=0, 02=1, etc.
            if 0 <= row < 16 and 0 <= col < 24:
                well_matrix[row, col] = count
    
    im = ax.imshow(well_matrix, cmap='YlOrRd', aspect='auto')
    ax.set_title('Profile Count by Well Position')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_xticks(range(0, 24, 4))
    ax.set_xticklabels(range(1, 25, 4))
    ax.set_yticks(range(16))
    ax.set_yticklabels([chr(ord('A') + i) for i in range(16)])
    plt.colorbar(im, ax=ax)
    
    # 2. Plate distribution
    ax = axes[1]
    plates = collected_profiles['Metadata_Plate'].to_list()
    plate_counts = Counter(plates)
    
    # Show top 20 plates
    top_plates = dict(plate_counts.most_common(20))
    ax.bar(range(len(top_plates)), list(top_plates.values()), color='skyblue')
    ax.set_title('Top 20 Plates by Profile Count')
    ax.set_xlabel('Plate (ranked)')
    ax.set_ylabel('Number of Profiles')
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    
    # 3. Source breakdown
    ax = axes[2]
    sources = collected_profiles['Metadata_Source'].to_list()
    source_counts = Counter(sources)
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(source_counts)))
    wedges, texts, autotexts = ax.pie(list(source_counts.values()), 
                                     labels=list(source_counts.keys()),
                                     autopct='%1.1f%%', colors=colors)
    ax.set_title('Profile Distribution by Source')
    
    # 4. Selected ORFs profile counts
    ax = axes[3]
    selected_counts = [Counter(collected_profiles['Metadata_JCP2022'].to_list())[orf] 
                      for orf in selected_orfs[:20]]  # First 20 selected ORFs
    
    ax.bar(range(len(selected_counts)), selected_counts, color='lightgreen')
    ax.set_title('Profile Counts for First 20 Selected ORFs')
    ax.set_xlabel('Selected ORF (index)')
    ax.set_ylabel('Number of Profiles')
    ax.axhline(y=5, color='red', linestyle='--', alpha=0.7, label='Expected (5)')
    ax.legend()
    
    # 5. Feature correlation matrix (sample)
    ax = axes[4]
    feature_cols = [col for col in collected_profiles.columns if col.startswith('X_')]
    sample_features = np.random.choice(feature_cols, min(10, len(feature_cols)), replace=False)
    
    corr_data = collected_profiles.select(sample_features).to_numpy()
    corr_matrix = np.corrcoef(corr_data.T)
    
    im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title('Feature Correlation Matrix (Sample)')
    ax.set_xticks(range(len(sample_features)))
    ax.set_yticks(range(len(sample_features)))
    ax.set_xticklabels([f'X_{i}' for i in range(len(sample_features))], rotation=45)
    ax.set_yticklabels([f'X_{i}' for i in range(len(sample_features))])
    plt.colorbar(im, ax=ax)
    
    # 6. Dataset summary comparison
    ax = axes[5]
    ax.axis('off')
    
    # Calculate some key statistics
    total_orfs = len(collected_profiles['Metadata_JCP2022'].unique())
    modal_profiles = 5  # We know this from earlier analysis
    selected_count = len(selected_orfs)
    
    summary_text = f"""
    Dataset Selection Summary
    ═══════════════════════
    
    Total Available ORFs:     {total_orfs:,}
    Modal Profiles per ORF:   {modal_profiles}
    
    Selected for Dataset:     {selected_count}
    Expected Total Images:    {selected_count * modal_profiles:,}
    
    Coverage:
    • {selected_count/total_orfs*100:.1f}% of available ORFs
    • Standardized to {modal_profiles} profiles each
    • Focus on high-quality, consistent data
    
    Next Steps:
    ✓ Validate image retrieval
    ✓ Create combined channel images
    ✓ Generate dataset metadata
    """
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, 
            fontsize=11, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    return fig

# Main execution function
def run_orf_dataset_visualization(profiles, selected_orfs):
    """Run all ORF dataset visualizations"""
    
    print("=== Creating ORF Dataset Visualizations ===")
    
    # 1. Dataset overview
    print("1. Generating dataset overview...")
    fig1 = visualize_orf_dataset_overview(profiles)
    plt.show()
    
    # 2. Sample images
    print("2. Loading sample ORF images...")
    fig2 = visualize_orf_sample_images(selected_orfs, num_samples=3)
    plt.show()
    
    # 3. Metadata distributions  
    print("3. Analyzing metadata distributions...")
    fig3 = visualize_orf_metadata_distributions(profiles, selected_orfs)
    plt.show()
    
    print("=== ORF Dataset Visualization Complete ===")
    
    return fig1, fig2, fig3

# Example usage:
# fig1, fig2, fig3 = run_orf_dataset_visualization(profiles, selected_orfs)

In [5]:
# Get sample of unique ORF perturbations
unique_orfs = profiles.select("Metadata_JCP2022").unique().collect().to_series()
print(f"\nTotal unique ORF perturbations: {len(unique_orfs)}")
print("Sample ORF perturbations:")
print(unique_orfs.sample(10, seed=42))

# Display basic statistics about the ORF dataset
print("\n=== ORF Dataset Structure ===")
sample_profiles = profiles.select(pl.col("^Metadata.*$")).limit(5).collect()
print(sample_profiles)

print("\n=== Checking for Image Location Metadata ===")
sample_with_all = profiles.limit(3).collect()
print("Full sample row (first 3 rows):")
for i, row in enumerate(sample_with_all.iter_rows(named=True)):
    print(f"Row {i}: {dict(row)}")
    if i == 0:  # Just show structure for first row
        break

# Check if we have the minimum required metadata for image retrieval
required_metadata = ["Metadata_Source", "Metadata_Plate", "Metadata_Well"]
missing_metadata = [col for col in required_metadata if col not in meta_cols]
if missing_metadata:
    print(f"⚠️  Warning: Missing required metadata: {missing_metadata}")
else:
    print("✅ All required metadata columns present")


Total unique ORF perturbations: 15131
Sample ORF perturbations:
shape: (10,)
Series: 'Metadata_JCP2022' [str]
[
	"JCP2022_904385"
	"JCP2022_900418"
	"JCP2022_910395"
	"JCP2022_902622"
	"JCP2022_903126"
	"JCP2022_901936"
	"JCP2022_902408"
	"JCP2022_902273"
	"JCP2022_903801"
	"JCP2022_906996"
]

=== ORF Dataset Structure ===
shape: (5, 4)
┌─────────────────┬────────────────┬───────────────┬──────────────────┐
│ Metadata_Source ┆ Metadata_Plate ┆ Metadata_Well ┆ Metadata_JCP2022 │
│ ---             ┆ ---            ┆ ---           ┆ ---              │
│ str             ┆ str            ┆ str           ┆ str              │
╞═════════════════╪════════════════╪═══════════════╪══════════════════╡
│ source_4        ┆ BR00117035     ┆ A01           ┆ JCP2022_905588   │
│ source_4        ┆ BR00117035     ┆ K16           ┆ JCP2022_904671   │
│ source_4        ┆ BR00117035     ┆ K15           ┆ JCP2022_910002   │
│ source_4        ┆ BR00117035     ┆ K14           ┆ JCP2022_915130   │
│ source_4  

## Notebook functions

In [6]:
def combine_channels_for_microsplit(
    channel_images,
    channels_to_combine=["DNA", "RNA", "ER", "AGP", "Mito"],
    weights=None,
    normalize=False
):
    """
    Combine multiple channels into a single image for μSplit input
    
    Parameters
    ----------
    channel_images: dict
        Dictionary containing channel names and their corresponding images
    channels_to_combine: list
        List of channel names to combine
    weights: list or None
        Optional weights for each channel
    normalize: bool
        Whether to normalize intensity values
    
    Returns
    --------
    combined_img: numpy.ndarray
        Combined image
    stats: dict
        Statistics about the combination process
    """
    processed_images = {}
    
    # Process each channel
    for channel in channels_to_combine:
        if channel not in channel_images:
            raise ValueError(f"Channel '{channel}' not found in channel_images")
        
        img = channel_images[channel].astype(np.float32)
        if normalize:
            # Avoid division by zero
            img_min, img_max = img.min(), img.max()
            if img_max > img_min:
                img = (img - img_min) / (img_max - img_min)
        
        processed_images[channel] = img
    
    # Apply weights for combining
    if weights is None:
        weights = [1.0 / len(channels_to_combine)] * len(channels_to_combine)
    
    if len(weights) != len(channels_to_combine):
        raise ValueError(f"Number of weights ({len(weights)}) must match number of channels ({len(channels_to_combine)})")
    
    # Create combined image
    combined_img = np.zeros_like(processed_images[channels_to_combine[0]])
    for channel, weight in zip(channels_to_combine, weights):
        combined_img += processed_images[channel] * weight
    
    # Collect statistics
    stats = {
        "min_val": combined_img.min(),
        "max_val": combined_img.max(),
        "mean_val": combined_img.mean(),
        "std_val": combined_img.std()
    }
    
    return combined_img, stats

# Add the visualization function that was missing
def display_orf_channels(source, batch, plate, well, site, orf_id, channels=None):
    """Display channels for an ORF sample"""
    if channels is None:
        channels = ["DNA", "RNA", "ER", "AGP", "Mito"]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    channel_images = {}
    
    for i, channel in enumerate(channels):
        try:
            img = get_jump_image(source, batch, plate, well, channel, site, None)
            channel_images[channel] = img
            
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f"{channel}")
            axes[i].axis('off')
        except Exception as e:
            axes[i].text(0.5, 0.5, f"Error loading\n{channel}", 
                        ha='center', va='center', transform=axes[i].transAxes)
    
    # Combined image in last subplot
    if len(channel_images) > 0:
        combined_img, _ = combine_channels_for_microsplit(channel_images)
        axes[5].imshow(combined_img, cmap='viridis')
        axes[5].set_title("Combined")
        axes[5].axis('off')
    
    plt.suptitle(f"ORF: {orf_id}")
    plt.tight_layout()
    
    return channel_images

# Simplified dataset creation function (we'll fix the complex one later)
def create_orf_dataset_simple(profiles, selected_orfs, channels=["DNA", "RNA", "ER", "AGP", "Mito"]):
    """Simple function to test dataset creation workflow"""
    print(f"Would create dataset with {len(selected_orfs)} ORFs and {len(channels)} channels")
    return {"status": "ready"}

print("Functions defined successfully!")

Functions defined successfully!


In [7]:
# Add these functions to complete your function definitions:

def select_orfs_for_dataset(profiles, target_images=300, seed=42):
    """Select ORFs with standard profile counts for dataset creation"""
    
    # Count profiles per ORF
    profile_counts = (
        profiles
        .select("Metadata_JCP2022")
        .value_counts()
        .rename({"count": "profile_count"})
    ).collect()
    
    # Find modal (most common) profile count
    modal_count = profile_counts.select("profile_count").mode().item()
    
    # Get ORFs with standard profile count
    standard_orfs = (
        profile_counts
        .filter(pl.col("profile_count") == modal_count)
        .select("Metadata_JCP2022")
        .to_series()
        .to_list()
    )
    
    # Calculate how many ORFs we need
    required_orfs = target_images // modal_count
    
    # Random selection
    np.random.seed(seed)
    selected = np.random.choice(standard_orfs, min(required_orfs, len(standard_orfs)), replace=False)
    
    return selected.tolist(), modal_count, len(standard_orfs)

def test_orf_image_retrieval(orf_id, profiles):
    """Test image retrieval for a single ORF"""
    try:
        # Get this ORF's profiles from the dataset
        orf_profiles = profiles.filter(
            pl.col("Metadata_JCP2022") == orf_id
        ).collect()
        
        if orf_profiles.shape[0] == 0:
            return False, f"No profiles found for {orf_id}"
            
        # Try to get image location info using jump-portrait
        test_info = get_item_location_info(orf_id)
        return True, f"Found {test_info.shape[0]} images via jump-portrait"
        
    except Exception as e:
        return False, str(e)

def get_orf_metadata_basic(selected_orfs):
    """Create basic metadata for selected ORFs"""
    return [
        {
            "orf_id": orf_id,
            "pert_type": "orf",
            "gene_name": "unknown"  # skip gene name lookup for now
        }
        for orf_id in selected_orfs
    ]

## ORF selection for dataset

In [8]:
# ORF Selection Strategy - Fixed Implementation
print("=== ORF Selection for Dataset Creation ===")

# Fix: Count profiles per ORF using collected data
print("Analyzing profile distribution...")
profile_counts = (
    profiles
    .group_by("Metadata_JCP2022")
    .agg(pl.len().alias("profile_count"))
    .collect()
)

print("Profile distribution:")
distribution = (
    profile_counts
    .group_by("profile_count")
    .agg(pl.len().alias("orf_count"))
    .sort("profile_count")
)  # No .collect() needed since profile_counts is already collected

total_orfs = profile_counts.height  # Use height instead of len(unique_orfs)
for row in distribution.iter_rows(named=True):
    count = row["profile_count"]
    freq = row["orf_count"]
    percentage = (freq / total_orfs) * 100
    print(f" {freq:,} ORFs ({percentage:.1f}%) have {count} profiles each")

# Use modal value (most common) for standard dataset creation
modal_profile_count = profile_counts.select("profile_count").to_series().mode()[0]
print(f"\nModal profile count: {modal_profile_count}")

# Dataset parameters
TARGET_IMAGES = 300
PROFILES_PER_ORF = modal_profile_count
REQUIRED_ORFS = TARGET_IMAGES // PROFILES_PER_ORF

print(f"\nDataset creation strategy:")
print(f"- Target images: {TARGET_IMAGES}")
print(f"- Profiles per ORF: {PROFILES_PER_ORF}")
print(f"- Required ORFs: {REQUIRED_ORFS}")
print(f"- Total images: {REQUIRED_ORFS * PROFILES_PER_ORF}")

# Select ORFs with the standard profile count
orfs_with_standard_profiles = (
    profile_counts
    .filter(pl.col("profile_count") == PROFILES_PER_ORF)
    .select("Metadata_JCP2022")
    .to_series()
    .to_list()
)

print(f"\nAvailable ORFs with {PROFILES_PER_ORF} profiles: {len(orfs_with_standard_profiles):,}")

# Random selection for reproducibility
np.random.seed(42)
selected_orfs = np.random.choice(orfs_with_standard_profiles, REQUIRED_ORFS, replace=False).tolist()

print(f"Selected {len(selected_orfs)} ORFs for dataset creation")
print(f"Sample selected ORFs: {selected_orfs[:10]}")

=== ORF Selection for Dataset Creation ===
Analyzing profile distribution...
Profile distribution:
 10 ORFs (0.1%) have 4 profiles each
 15,030 ORFs (99.3%) have 5 profiles each
 86 ORFs (0.6%) have 10 profiles each
 4 ORFs (0.0%) have 920 profiles each
 1 ORFs (0.0%) have 1930 profiles each

Modal profile count: 5

Dataset creation strategy:
- Target images: 300
- Profiles per ORF: 5
- Required ORFs: 60
- Total images: 300

Available ORFs with 5 profiles: 15,030
Selected 60 ORFs for dataset creation
Sample selected ORFs: ['JCP2022_904597', 'JCP2022_906196', 'JCP2022_907074', 'JCP2022_915023', 'JCP2022_914858', 'JCP2022_902085', 'JCP2022_902604', 'JCP2022_901522', 'JCP2022_900909', 'JCP2022_912054']


15,030 ORFs (99.3%) have exactly 5 profiles each

96 ORFs have 4-10 profiles (likely experimental variations)

1 ORF has 1,930 profiles (clear outlier, possibly a control or highly replicated target)

In [9]:
# ORF Image Retrieval Diagnostics
print("=== Diagnosing ORF Image Access Issues ===")

# Let's examine the actual data structure to understand the problem
test_orf = selected_orfs[0]
print(f"Diagnosing ORF: {test_orf}")

# Get the full profile data for this ORF
orf_profiles = profiles.filter(pl.col("Metadata_JCP2022") == test_orf).collect()
print(f"Found {orf_profiles.height} profiles")

# Let's examine what metadata we actually have
print("\nAvailable metadata columns:")
for col in orf_profiles.columns:
    if col.startswith("Metadata"):
        sample_values = orf_profiles.select(col).unique().to_series().to_list()[:5]
        print(f"  {col}: {sample_values}")

# Compare with a known working example from CRISPR data
print("\n=== Comparing with Reference Data Structure ===")
print("The CRISPR data uses get_item_location_info(gene_name) which returns:")
print("- Metadata_Source, Metadata_Batch, Metadata_Plate, Metadata_Well, Metadata_Site")
print("- But ORF profiles only have: Metadata_Source, Metadata_Plate, Metadata_Well")

# Alternative approach 1: Try to find actual gene names for ORF IDs
print("\n=== Attempting Gene Name Resolution ===")

def try_gene_name_lookup(jcp_id):
    """Try different approaches to find the gene name for JCP ID"""
    try:
        # Method 1: Check if broad_babel can help with mapping
        from broad_babel.query import get_mapper
        try:
            # Try different mapper configurations
            mapper = get_mapper(
                query="JCP2022_to_gene",  # Hypothetical mapping
                input_column="JCP2022", 
                output_columns=["gene_symbol"]
            )
            result = mapper.query(jcp_id)
            if result:
                return result.get('gene_symbol', None)
        except:
            pass
            
        # Method 2: Try jump-portrait directly with JCP ID
        try:
            info = get_item_location_info(jcp_id)
            return f"Direct success: {info.shape[0]} images"
        except Exception as e:
            pass
            
        # Method 3: Search for patterns in the JCP ID
        # Sometimes JCP IDs contain encoded gene information
        return None
        
    except Exception as e:
        return f"Error: {str(e)}"

# Test gene name resolution
for i, orf in enumerate(selected_orfs[:3]):
    result = try_gene_name_lookup(orf)
    print(f"  {orf}: {result}")

# Alternative approach 2: Try different get_jump_image parameters
print("\n=== Testing Different Image Access Methods ===")

def test_image_access_methods(source, plate, well):
    """Test different parameter combinations for ORF image access"""
    methods = [
        # (batch, site, description)
        ("unknown", 1, "Default unknown batch, site 1"),
        (None, None, "All None"),
        ("", "", "Empty strings"),
        ("batch", "site", "Generic strings"),
        ("ORF", 0, "ORF batch, site 0"),
        ("source_4_batch", 1, "Constructed batch name"),
    ]
    
    for batch, site, desc in methods:
        try:
            img = get_jump_image(source, batch, plate, well, "DNA", site, None)
            return True, f"SUCCESS with {desc}: {img.shape}"
        except Exception as e:
            continue
    
    return False, "All methods failed"

# Test with first ORF location
first_orf_data = orf_profiles.row(0)
source = first_orf_data[orf_profiles.columns.index("Metadata_Source")]
plate = first_orf_data[orf_profiles.columns.index("Metadata_Plate")]
well = first_orf_data[orf_profiles.columns.index("Metadata_Well")]

success, message = test_image_access_methods(source, plate, well)
print(f"Image access test: {message}")

# Alternative approach 3: Check if we need to construct file paths differently
print("\n=== Investigating Data Storage Structure ===")
print("The 404 error suggests the images are either:")
print("1. Stored in a different location structure than expected")
print("2. Require different batch/site parameter handling")
print("3. Need a different API endpoint or access method")
print("4. The ORF dataset uses a different storage system entirely")

# Recommend next steps based on findings
print("\n=== Recommended Next Steps ===")
print("1. Check if ORF images are stored in AWS S3 with different path structure")
print("2. Consult JUMP documentation for ORF-specific image access patterns") 
print("3. Try using jump-portrait with different query methods for ORF data")
print("4. Consider if ORF images need different credentials/access permissions")

# Fallback: Create a simplified dataset using profile metadata only
print("\n=== Fallback Option ===")
print("If image access cannot be resolved, we can:")
print("1. Create a metadata-only dataset for testing pipeline structure")
print("2. Use synthetic/placeholder images to validate the pipeline")
print("3. Focus on fixing the image access in parallel")

def create_metadata_only_dataset(selected_orfs, profiles, output_dir="orf_metadata_only"):
    """Create dataset structure with metadata but no images for pipeline testing"""
    import pandas as pd
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    metadata_list = []
    image_counter = 0
    
    for orf_id in selected_orfs:
        orf_data = profiles.filter(pl.col("Metadata_JCP2022") == orf_id).collect()
        
        for i in range(min(5, orf_data.height)):  # 5 images per ORF
            row = orf_data.row(i)
            metadata_list.append({
                "image_id": image_counter,
                "orf_perturbation": orf_id,
                "source": row[orf_data.columns.index("Metadata_Source")],
                "plate": row[orf_data.columns.index("Metadata_Plate")],
                "well": row[orf_data.columns.index("Metadata_Well")],
                "batch": "ORF_batch",  # Placeholder
                "site": 1,  # Placeholder
                "combined_channels": ["DNA", "RNA", "ER", "AGP", "Mito"],
                "pert_type": "orf",
                "dataset": "orf_metadata_test"
            })
            image_counter += 1
    
    # Save metadata
    df = pd.DataFrame(metadata_list)
    df.to_csv(os.path.join(output_dir, "metadata_structure_test.csv"), index=False)
    
    return df

print("Run create_metadata_only_dataset() to generate pipeline structure for testing")

=== Diagnosing ORF Image Access Issues ===
Diagnosing ORF: JCP2022_904597
Found 5 profiles

Available metadata columns:
  Metadata_Source: ['source_4']
  Metadata_Plate: ['BR00123790', 'BR00123791', 'BR00123786', 'BR00123787', 'BR00123785']
  Metadata_Well: ['M09']
  Metadata_JCP2022: ['JCP2022_904597']

=== Comparing with Reference Data Structure ===
The CRISPR data uses get_item_location_info(gene_name) which returns:
- Metadata_Source, Metadata_Batch, Metadata_Plate, Metadata_Well, Metadata_Site
- But ORF profiles only have: Metadata_Source, Metadata_Plate, Metadata_Well

=== Attempting Gene Name Resolution ===
  JCP2022_904597: None
  JCP2022_906196: None
  JCP2022_907074: None

=== Testing Different Image Access Methods ===
Image access test: All methods failed

=== Investigating Data Storage Structure ===
The 404 error suggests the images are either:
1. Stored in a different location structure than expected
2. Require different batch/site parameter handling
3. Need a different AP

In [10]:
# Intermediate Solution: Build pipeline structure while solving image access
print("=== Creating ORF Dataset Structure (Intermediate Approach) ===")

import pandas as pd
import os

def create_orf_pipeline_structure(selected_orfs, profiles, output_dir="orf_microsplit_dataset"):
    """
    Create the complete pipeline structure with proper metadata
    This allows testing of downstream pipeline while image access is resolved
    """
    
    # Create directory structure matching reference notebooks
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "combined"), exist_ok=True)
    
    channels = ["DNA", "RNA", "ER", "AGP", "Mito"]
    for channel in channels:
        os.makedirs(os.path.join(output_dir, channel), exist_ok=True)
    
    # Generate comprehensive metadata following pilot notebook format
    metadata_list = []
    image_counter = 0
    
    print(f"Processing {len(selected_orfs)} ORFs...")
    
    for orf_idx, orf_id in enumerate(selected_orfs):
        # Get ORF profiles
        orf_data = profiles.filter(pl.col("Metadata_JCP2022") == orf_id).collect()
        
        if orf_data.height == 0:
            continue
            
        # Process 5 profiles per ORF (matching your modal analysis)
        for i in range(min(5, orf_data.height)):
            row = orf_data.row(i)
            
            # Extract metadata following reference notebook pattern
            metadata_entry = {
                # Core identifiers
                "image_id": image_counter,
                "orf_perturbation": orf_id,
                "pert_type": "orf",
                
                # Location metadata (from profiles)
                "source": row[orf_data.columns.index("Metadata_Source")],
                "plate": row[orf_data.columns.index("Metadata_Plate")],
                "well": row[orf_data.columns.index("Metadata_Well")],
                
                # ORF-specific handling for missing metadata
                "batch": f"ORF_batch_{orf_idx//10}",  # Group ORFs into batches
                "site": 1,  # Default site
                
                # Channel information
                "combined_channels": channels,
                "weights": [0.2, 0.2, 0.2, 0.2, 0.2],  # Equal weights
                "normalized": False,
                
                # Dataset information
                "dataset": "orf",
                "experiment": "microsplit_orf",
                "channel_combination": "_".join(channels),
                
                # File paths (to be populated when images are available)
                "combined_path": f"combined/img_{image_counter:05d}_combined.tif",
                "dna_path": f"DNA/img_{image_counter:05d}_DNA.tif",
                "rna_path": f"RNA/img_{image_counter:05d}_RNA.tif", 
                "er_path": f"ER/img_{image_counter:05d}_ER.tif",
                "agp_path": f"AGP/img_{image_counter:05d}_AGP.tif",
                "mito_path": f"Mito/img_{image_counter:05d}_Mito.tif",
                
                # Placeholder for stats (to be filled when images processed)
                "min_val": None,
                "max_val": None,
                "mean_val": None,
                "std_val": None
            }
            
            metadata_list.append(metadata_entry)
            image_counter += 1
    
    # Create metadata DataFrame
    metadata_df = pd.DataFrame(metadata_list)
    
    # Save metadata CSV
    metadata_path = os.path.join(output_dir, "dataset_metadata.csv")
    metadata_df.to_csv(metadata_path, index=False)
    
    # Create summary info
    summary = {
        "total_images": len(metadata_list),
        "total_orfs": len(selected_orfs),
        "successful_orfs": metadata_df['orf_perturbation'].nunique(),
        "output_directory": output_dir,
        "metadata_file": metadata_path,
        "channels": channels,
        "structure_complete": True,
        "images_ready": False  # Flag for image processing status
    }
    
    # Save summary JSON
    import json
    with open(os.path.join(output_dir, "dataset_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"✓ Dataset structure created:")
    print(f"  - Output directory: {output_dir}")
    print(f"  - Metadata file: {metadata_path}")
    print(f"  - Total images planned: {summary['total_images']}")
    print(f"  - Successfully processed ORFs: {summary['successful_orfs']}")
    print(f"  - Channel directories: {', '.join(channels)}")
    
    return summary, metadata_df

# Create the pipeline structure
summary, metadata_df = create_orf_pipeline_structure(selected_orfs, profiles)

# Display metadata sample
print(f"\nMetadata structure preview:")
print(f"Shape: {metadata_df.shape}")
print(f"Columns: {list(metadata_df.columns)}")
print(f"\nSample rows:")
print(metadata_df[['image_id', 'orf_perturbation', 'source', 'plate', 'well']].head())

# Create image processing function for when access is resolved
def process_images_when_ready(metadata_df, output_dir):
    """
    Function to process actual images once access method is determined
    This will populate the image files and update metadata stats
    """
    print("Image processing function created")
    print("Run this when image access is resolved:")
    print("process_images_when_ready(metadata_df, 'orf_microsplit_dataset')")

print(f"\n=== Next Steps ===")
print(f"1. ✓ Dataset structure created with proper metadata")
print(f"2. ⏳ Solve ORF image access (continue investigating)")
print(f"3. ⏳ Process actual images using process_images_when_ready()")
print(f"4. ✓ Pipeline ready for 01_noisemodels testing (with structure)")

# Verify compatibility with downstream pipeline
print(f"\n=== Pipeline Compatibility Check ===")
required_columns = ['image_id', 'source', 'batch', 'plate', 'well', 'site', 'combined_channels']
missing_columns = [col for col in required_columns if col not in metadata_df.columns]

if not missing_columns:
    print("✓ Metadata structure compatible with downstream pipeline")
else:
    print(f"⚠ Missing columns for pipeline: {missing_columns}")

print("\nDataset structure ready. Continue investigating ORF image access in parallel.")

=== Creating ORF Dataset Structure (Intermediate Approach) ===
Processing 60 ORFs...
✓ Dataset structure created:
  - Output directory: orf_microsplit_dataset
  - Metadata file: orf_microsplit_dataset/dataset_metadata.csv
  - Total images planned: 300
  - Successfully processed ORFs: 60
  - Channel directories: DNA, RNA, ER, AGP, Mito

Metadata structure preview:
Shape: (300, 24)
Columns: ['image_id', 'orf_perturbation', 'pert_type', 'source', 'plate', 'well', 'batch', 'site', 'combined_channels', 'weights', 'normalized', 'dataset', 'experiment', 'channel_combination', 'combined_path', 'dna_path', 'rna_path', 'er_path', 'agp_path', 'mito_path', 'min_val', 'max_val', 'mean_val', 'std_val']

Sample rows:
   image_id orf_perturbation    source       plate well
0         0   JCP2022_904597  source_4  BR00123785  M09
1         1   JCP2022_904597  source_4  BR00123786  M09
2         2   JCP2022_904597  source_4  BR00123787  M09
3         3   JCP2022_904597  source_4  BR00123790  M09
4       

In [11]:
# ORF Image Access Strategy Based on JUMP FAQ Insights
print("=== Updated ORF Image Access Strategy ===")
print("Key insight from JUMP FAQ: ORF profiles aggregate by Metadata_NCBI_Gene_ID, not JCP2022")

# Let's check if our ORF profiles have NCBI Gene ID metadata
sample_orf = selected_orfs[0]
orf_sample_data = profiles.filter(pl.col("Metadata_JCP2022") == sample_orf).collect()

print(f"Examining metadata for sample ORF: {sample_orf}")
print("Available metadata columns:")
metadata_cols = [col for col in orf_sample_data.columns if col.startswith("Metadata")]
for col in metadata_cols:
    sample_val = orf_sample_data.select(col).unique().to_series().to_list()[0]
    print(f"  {col}: {sample_val}")

# Check if NCBI Gene ID exists
has_ncbi_gene_id = "Metadata_NCBI_Gene_ID" in metadata_cols
print(f"\nHas Metadata_NCBI_Gene_ID: {has_ncbi_gene_id}")

if has_ncbi_gene_id:
    # Strategy 1: Try using NCBI Gene IDs for image access
    print("\n=== Testing Gene ID-based Image Access ===")
    
    def test_gene_id_image_access(profiles, orf_jcp_id):
        """Test image access using NCBI Gene ID instead of JCP ID"""
        try:
            # Get the NCBI Gene ID for this JCP ID
            orf_data = profiles.filter(pl.col("Metadata_JCP2022") == orf_jcp_id).collect()
            
            if orf_data.height == 0:
                return False, "No profile data found"
                
            # Extract NCBI Gene ID
            ncbi_gene_id = orf_data.select("Metadata_NCBI_Gene_ID").unique().to_series().to_list()[0]
            print(f"  JCP ID {orf_jcp_id} -> NCBI Gene ID: {ncbi_gene_id}")
            
            # Try jump-portrait with Gene ID
            try:
                gene_info = get_item_location_info(str(ncbi_gene_id))
                return True, f"Success with Gene ID: {gene_info.shape[0]} images found"
            except Exception as e:
                return False, f"Gene ID lookup failed: {str(e)}"
                
        except Exception as e:
            return False, f"Error: {str(e)}"
    
    # Test with first few ORFs
    print("Testing Gene ID approach:")
    for orf in selected_orfs[:3]:
        success, message = test_gene_id_image_access(profiles, orf)
        print(f"  {orf}: {message}")

else:
    print("No NCBI Gene ID found - need alternative approach")

# Strategy 2: Check for gene symbol mapping using broad_babel
print("\n=== Testing broad_babel Mapping Approach ===")

def test_broad_babel_mapping():
    """Test if broad_babel can map JCP2022 IDs to gene symbols"""
    try:
        from broad_babel.query import get_mapper
        
        # Try different broad_babel configurations for ORF mapping
        test_configs = [
            ("standard_key", "JCP2022", ["gene_symbol"]),
            ("pert_iname", "JCP2022", ["standard_key"]), 
            ("JCP2022", "standard_key", ["gene_symbol"]),
        ]
        
        for input_col, query_col, output_cols in test_configs:
            try:
                print(f"  Trying mapper: {input_col} -> {output_cols}")
                mapper = get_mapper(
                    query=f"{input_col}_to_{output_cols[0]}", 
                    input_column=query_col,
                    output_columns=output_cols
                )
                
                # Test with first ORF
                test_result = mapper.query(selected_orfs[0])
                if test_result:
                    print(f"    SUCCESS: {selected_orfs[0]} -> {test_result}")
                    return mapper, True
                else:
                    print(f"    No result for {selected_orfs[0]}")
                    
            except Exception as e:
                print(f"    Failed: {str(e)}")
                continue
                
        return None, False
        
    except Exception as e:
        print(f"  broad_babel import/setup failed: {str(e)}")
        return None, False

mapper, babel_success = test_broad_babel_mapping()

# Strategy 3: Alternative direct approach using profile metadata
print("\n=== Testing Direct Profile-Based Image Construction ===")

def construct_orf_image_path(profile_row, channel):
    """
    Attempt to construct image paths directly from profile metadata
    Based on JUMP image storage patterns
    """
    try:
        source = profile_row["Metadata_Source"]
        plate = profile_row["Metadata_Plate"] 
        well = profile_row["Metadata_Well"]
        
        # JUMP images are typically stored as:
        # s3://cellpainting-gallery/cpg0016-jump/{source}/images/{plate}/{well}/*.tiff
        # We need to determine the exact pattern for ORF data
        
        potential_paths = [
            f"{source}/{plate}/{well}/{channel}",
            f"{source}/images/{plate}/{well}_{channel}",
            f"cpg0016-jump/{source}/images/{plate}/Images/{well}*{channel}*",
        ]
        
        return potential_paths
        
    except Exception as e:
        return [f"Path construction failed: {str(e)}"]

# Test path construction
sample_row = orf_sample_data.row(0, named=True)
test_paths = construct_orf_image_path(sample_row, "DNA")
print("Potential image paths for ORF data:")
for path in test_paths:
    print(f"  {path}")

# Strategy 4: Check if ORF images require different jump-portrait setup
print("\n=== Investigating jump-portrait Configuration ===")
print("ORF images might require:")
print("1. Different AWS credentials or access permissions")
print("2. Different jump-portrait backend configuration") 
print("3. Specific ORF dataset setup in jump-portrait")
print("4. Images stored in different S3 buckets/paths")

# Recommended diagnostic steps
print("\n=== Recommended Next Steps for Image Access ===")
print("1. Check if your ORF profiles have Metadata_NCBI_Gene_ID")
print("2. Try jump-portrait with gene symbols instead of JCP IDs")
print("3. Contact JUMP consortium about ORF-specific image access")
print("4. Check if jump-portrait needs ORF-specific configuration")
print("5. Look for ORF examples in jump-portrait documentation/issues")

# Update the dataset creation to handle multiple access strategies
def create_flexible_orf_dataset(selected_orfs, profiles, output_dir="orf_microsplit_dataset"):
    """
    Updated dataset creation that tries multiple image access strategies
    """
    print("Creating flexible ORF dataset with multiple access strategies...")
    
    # Try each strategy for each ORF
    strategies = [
        ("jcp_direct", lambda orf: get_item_location_info(orf)),
        ("gene_id", lambda orf: test_gene_id_image_access(profiles, orf) if has_ncbi_gene_id else (False, "No Gene ID")),
        ("babel_mapping", lambda orf: None)  # To be implemented based on babel success
    ]
    
    successful_access = {}
    failed_access = []
    
    for orf in selected_orfs[:5]:  # Test first 5
        orf_success = False
        for strategy_name, strategy_func in strategies:
            try:
                if strategy_name == "jcp_direct":
                    result = strategy_func(orf)
                    if result.shape[0] > 0:
                        successful_access[orf] = (strategy_name, result)
                        orf_success = True
                        break
                elif strategy_name == "gene_id":
                    success, message = strategy_func(orf)
                    if success:
                        successful_access[orf] = (strategy_name, message)
                        orf_success = True
                        break
            except:
                continue
                
        if not orf_success:
            failed_access.append(orf)
    
    print(f"Access test results:")
    print(f"  Successful: {len(successful_access)}")
    print(f"  Failed: {len(failed_access)}")
    
    return successful_access, failed_access

print("\nRun create_flexible_orf_dataset() to test multiple access strategies")

=== Updated ORF Image Access Strategy ===
Key insight from JUMP FAQ: ORF profiles aggregate by Metadata_NCBI_Gene_ID, not JCP2022
Examining metadata for sample ORF: JCP2022_904597
Available metadata columns:
  Metadata_Source: source_4
  Metadata_Plate: BR00123787
  Metadata_Well: M09
  Metadata_JCP2022: JCP2022_904597

Has Metadata_NCBI_Gene_ID: False
No NCBI Gene ID found - need alternative approach

=== Testing broad_babel Mapping Approach ===
  Trying mapper: standard_key -> ['gene_symbol']
    Failed: 'list' object has no attribute 'split'
  Trying mapper: pert_iname -> ['standard_key']
    Failed: 'list' object has no attribute 'split'
  Trying mapper: JCP2022 -> ['gene_symbol']
    Failed: 'list' object has no attribute 'split'

=== Testing Direct Profile-Based Image Construction ===
Potential image paths for ORF data:
  source_4/BR00123785/M09/DNA
  source_4/images/BR00123785/M09_DNA
  cpg0016-jump/source_4/images/BR00123785/Images/M09*DNA*

=== Investigating jump-portrait Conf

In [12]:
# Check what metadata your ORF profiles actually contain
sample_orf = selected_orfs[0]
orf_sample_data = profiles.filter(pl.col("Metadata_JCP2022") == sample_orf).collect()
metadata_cols = [col for col in orf_sample_data.columns if col.startswith("Metadata")]
print("Available metadata columns:")
for col in metadata_cols:
    print(f"  {col}")

Available metadata columns:
  Metadata_Source
  Metadata_Plate
  Metadata_Well
  Metadata_JCP2022


## oh no aggregated profiles instead of raw data! let's switch...

In [13]:
# ORF Dataset Solution: Access Raw Image Metadata
print("=== Accessing Raw ORF Image Metadata ===")

# The issue: You're using processed profiles instead of raw metadata
# Solution: Load the raw metadata that contains full image location info

import requests
import pandas as pd
from io import StringIO

def load_jump_experimental_metadata():
    """Load the experimental metadata that contains batch information"""
    try:
        # JUMP experimental metadata from GitHub
        metadata_url = "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/experimental-metadata.tsv"
        response = requests.get(metadata_url)
        response.raise_for_status()
        
        # Parse TSV
        experimental_metadata = pd.read_csv(StringIO(response.text), sep='\t')
        print(f"Loaded experimental metadata: {experimental_metadata.shape}")
        
        # Filter for ORF experiments
        orf_experiments = experimental_metadata[experimental_metadata['Perturbation'] == 'orf']
        print(f"ORF experiments found: {len(orf_experiments)}")
        
        if len(orf_experiments) > 0:
            print("ORF experiment details:")
            print(orf_experiments[['Batch', 'Assay_Plate_Barcode', 'Perturbation', 'Cell_type']].head())
            
        return experimental_metadata, orf_experiments
        
    except Exception as e:
        print(f"Failed to load experimental metadata: {str(e)}")
        return None, None

# Load experimental metadata
experimental_metadata, orf_experiments = load_jump_experimental_metadata()

def find_orf_load_data_files():
    """Find the load_data CSV files that contain full image metadata"""
    
    if orf_experiments is None or len(orf_experiments) == 0:
        print("No ORF experiments found - cannot locate load_data files")
        return []
        
    # Get unique ORF batches
    orf_batches = orf_experiments['Batch'].unique()
    print(f"ORF batches: {list(orf_batches)}")
    
    # Try to construct load_data URLs based on JUMP structure
    load_data_candidates = []
    
    for batch in orf_batches:
        batch_plates = orf_experiments[orf_experiments['Batch'] == batch]['Assay_Plate_Barcode'].tolist()
        
        for plate in batch_plates[:3]:  # Test first 3 plates
            # Based on project knowledge structure
            potential_urls = [
                f"https://github.com/jump-cellpainting/datasets/raw/main/load_data_csv/{batch}/{plate}/load_data.csv",
                f"https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump/source_4/workspace/load_data_csv/{batch}/{plate}/load_data.csv.gz",
            ]
            
            for url in potential_urls:
                load_data_candidates.append({
                    'batch': batch,
                    'plate': plate, 
                    'url': url
                })
    
    print(f"Generated {len(load_data_candidates)} load_data candidates to test")
    return load_data_candidates

# Find load_data files
load_data_candidates = find_orf_load_data_files()

def test_load_data_access(candidates):
    """Test access to load_data files"""
    successful_files = []
    
    for candidate in candidates[:5]:  # Test first 5
        try:
            print(f"Testing: {candidate['url']}")
            response = requests.head(candidate['url'], timeout=10)
            
            if response.status_code == 200:
                print(f"  ✓ Accessible: {candidate['batch']}/{candidate['plate']}")
                successful_files.append(candidate)
            else:
                print(f"  ✗ Status {response.status_code}")
                
        except Exception as e:
            print(f"  ✗ Failed: {str(e)}")
            
    return successful_files

# Test load_data access
successful_load_data = test_load_data_access(load_data_candidates)

def load_sample_load_data(file_info):
    """Load a sample load_data file to understand structure"""
    try:
        response = requests.get(file_info['url'])
        response.raise_for_status()
        
        # Handle compressed files
        if file_info['url'].endswith('.gz'):
            import gzip
            content = gzip.decompress(response.content).decode('utf-8')
        else:
            content = response.text
            
        # Parse CSV
        load_data_df = pd.read_csv(StringIO(content))
        
        print(f"Load data structure for {file_info['batch']}/{file_info['plate']}:")
        print(f"  Shape: {load_data_df.shape}")
        print(f"  Columns: {list(load_data_df.columns)}")
        
        # Look for image location columns
        image_cols = [col for col in load_data_df.columns if 'Image' in col or 'Path' in col]
        if image_cols:
            print(f"  Image columns: {image_cols}")
            
        # Look for metadata columns  
        metadata_cols = [col for col in load_data_df.columns if col.startswith('Metadata')]
        if metadata_cols:
            print(f"  Metadata columns: {metadata_cols}")
            
        return load_data_df
        
    except Exception as e:
        print(f"Failed to load {file_info['url']}: {str(e)}")
        return None

# Load sample load_data if available
if successful_load_data:
    sample_load_data = load_sample_load_data(successful_load_data[0])
else:
    print("No accessible load_data files found")
    sample_load_data = None

# Alternative approach: Use JUMP datasets repository structure
def try_jump_datasets_approach():
    """Try accessing ORF metadata through jump datasets repository"""
    print("\n=== Alternative: JUMP Datasets Repository Approach ===")
    
    try:
        # Try to find ORF-specific metadata files
        base_url = "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata"
        
        metadata_files_to_try = [
            "orf_metadata.tsv",
            "orf_platemap.tsv", 
            "platemap_orf.tsv",
            "well_metadata.tsv"
        ]
        
        for filename in metadata_files_to_try:
            try:
                url = f"{base_url}/{filename}"
                response = requests.head(url)
                if response.status_code == 200:
                    print(f"  ✓ Found: {url}")
                    # Load and examine
                    data_response = requests.get(url)
                    df = pd.read_csv(StringIO(data_response.text), sep='\t')
                    print(f"    Shape: {df.shape}")
                    print(f"    Columns: {list(df.columns)[:10]}")  # First 10 columns
                    return df
                else:
                    print(f"  ✗ Not found: {filename}")
            except:
                continue
                
        return None
        
    except Exception as e:
        print(f"Repository approach failed: {str(e)}")
        return None

# Try repository approach
repo_metadata = try_jump_datasets_approach()

# Summary and next steps
print(f"\n=== ORF Image Access Investigation Summary ===")
print(f"✓ Problem identified: Using processed profiles instead of raw metadata")
print(f"✓ Experimental metadata: {'Available' if experimental_metadata is not None else 'Failed'}")
print(f"✓ ORF experiments found: {len(orf_experiments) if orf_experiments is not None else 0}")
print(f"✓ Load data files tested: {len(successful_load_data)} accessible")
print(f"✓ Repository metadata: {'Found' if repo_metadata is not None else 'Not found'}")

if successful_load_data or repo_metadata is not None:
    print(f"\n🎯 SOLUTION PATH IDENTIFIED:")
    print(f"You need to use the raw metadata files instead of processed profiles")
    print(f"This will provide the batch/site information needed for get_jump_image()")
else:
    print(f"\n⚠ ALTERNATIVE APPROACH NEEDED:")
    print(f"Raw metadata access requires further investigation")
    print(f"Consider contacting JUMP consortium for ORF-specific guidance")

# Provide the corrected approach
print(f"\n=== Corrected Approach for ORF Dataset Creation ===")
print(f"Instead of loading:")
print(f"  profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony.parquet")
print(f"")
print(f"You should load:")
print(f"  1. Raw load_data CSV files (contain full image metadata)")
print(f"  2. Experimental metadata TSV (contains batch information)")
print(f"  3. Platemap files (link JCP2022 IDs to well positions)")
print(f"")
print(f"This will provide the complete metadata needed for image retrieval")

=== Accessing Raw ORF Image Metadata ===
Failed to load experimental metadata: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/experimental-metadata.tsv
No ORF experiments found - cannot locate load_data files
No accessible load_data files found

=== Alternative: JUMP Datasets Repository Approach ===
  ✗ Not found: orf_metadata.tsv
  ✗ Not found: orf_platemap.tsv
  ✗ Not found: platemap_orf.tsv
  ✗ Not found: well_metadata.tsv

=== ORF Image Access Investigation Summary ===
✓ Problem identified: Using processed profiles instead of raw metadata
✓ Experimental metadata: Failed
✓ ORF experiments found: 0
✓ Load data files tested: 0 accessible
✓ Repository metadata: Not found

⚠ ALTERNATIVE APPROACH NEEDED:
Raw metadata access requires further investigation
Consider contacting JUMP consortium for ORF-specific guidance

=== Corrected Approach for ORF Dataset Creation ===
Instead of loading:
  profiles_wellpos_cc_var_mad_outlier

In [14]:
# Use the most common profile count (modal value) instead of maximum
print("=== Corrected ORF Selection Strategy ===")

# The majority of ORFs have 5 profiles - use this as our standard
PROFILES_PER_ORF = 5  # Modal value, not maximum
TARGET_IMAGES = 300
REQUIRED_ORFS = TARGET_IMAGES // PROFILES_PER_ORF  # 60 ORFs

print(f"Corrected strategy:")
print(f"- Profiles per ORF: {PROFILES_PER_ORF}")
print(f"- Required ORFs: {REQUIRED_ORFS}")  
print(f"- Total images: {REQUIRED_ORFS * PROFILES_PER_ORF}")

# Select ORFs with exactly 5 profiles (the standard case)
orfs_with_standard_profiles = (
    profile_counts
    .filter(pl.col("profile_count") == 5)
    .select("Metadata_JCP2022")
    .to_series()
    .to_list()
)

print(f"ORFs with {PROFILES_PER_ORF} profiles: {len(orfs_with_standard_profiles)}")

# Random selection from the 15,030 available ORFs
np.random.seed(42)
selected_orfs = np.random.choice(orfs_with_standard_profiles, REQUIRED_ORFS, replace=False).tolist()

print(f"Selected {len(selected_orfs)} ORFs")
print(f"First 10 selected ORFs: {selected_orfs[:10]}")

=== Corrected ORF Selection Strategy ===
Corrected strategy:
- Profiles per ORF: 5
- Required ORFs: 60
- Total images: 300
ORFs with 5 profiles: 15030
Selected 60 ORFs
First 10 selected ORFs: ['JCP2022_904597', 'JCP2022_906196', 'JCP2022_907074', 'JCP2022_915023', 'JCP2022_914858', 'JCP2022_902085', 'JCP2022_902604', 'JCP2022_901522', 'JCP2022_900909', 'JCP2022_912054']


In [15]:
# ORF Dataset Visualization - Similar to Pilot Dataset Exploration
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def visualize_orf_dataset_overview(profiles):
    """Create comprehensive overview visualizations of the ORF dataset"""
    
    # Collect basic data
    collected_profiles = profiles.collect()
    total_profiles = len(collected_profiles)
    unique_sources = collected_profiles['Metadata_Source'].unique()
    unique_plates = collected_profiles['Metadata_Plate'].unique()
    unique_orfs = collected_profiles['Metadata_JCP2022'].unique()
    
    # Count profiles per ORF
    orf_counts = Counter(collected_profiles['Metadata_JCP2022'].to_list())
    profile_distribution = Counter(orf_counts.values())
    
    # Create comprehensive figure
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Dataset Overview (top section)
    gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)
    
    # Dataset statistics
    ax_stats = fig.add_subplot(gs[0, :2])
    ax_stats.axis('off')
    
    stats_text = f"""
    JUMP ORF Dataset Overview
    ════════════════════════
    
    Total Profiles:     {total_profiles:,}
    Unique ORFs:        {len(unique_orfs):,}
    Data Sources:       {len(unique_sources)}
    Unique Plates:      {len(unique_plates):,}
    Feature Columns:    {len(collected_profiles.columns) - 4:,}
    Metadata Columns:   4
    
    Profile Distribution per ORF:
    • {profile_distribution[5]:,} ORFs ({profile_distribution[5]/len(unique_orfs)*100:.1f}%) have 5 profiles
    • {sum(v for k, v in profile_distribution.items() if k != 5):,} ORFs have other counts
    """
    
    ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes, 
                  fontsize=12, verticalalignment='top', fontfamily='monospace')
    
    # 2. Profile count distribution histogram
    ax_hist = fig.add_subplot(gs[0, 2:])
    counts = list(orf_counts.values())
    bins = np.arange(1, max(counts) + 2) - 0.5
    ax_hist.hist(counts, bins=bins, alpha=0.7, color='steelblue', edgecolor='black')
    ax_hist.set_xlabel('Number of Profiles per ORF')
    ax_hist.set_ylabel('Number of ORFs')
    ax_hist.set_title('Distribution of Profile Counts per ORF')
    ax_hist.grid(True, alpha=0.3)
    
    # Add text annotations for key statistics
    ax_hist.axvline(5, color='red', linestyle='--', alpha=0.8)
    ax_hist.text(5.2, ax_hist.get_ylim()[1]*0.8, f'Modal: 5 profiles\n({profile_distribution[5]:,} ORFs)', 
                 fontsize=10, color='red')
    
    # 3. Source and Plate distribution
    ax_sources = fig.add_subplot(gs[1, 0])
    source_counts = Counter(collected_profiles['Metadata_Source'].to_list())
    ax_sources.bar(range(len(source_counts)), list(source_counts.values()), color='lightcoral')
    ax_sources.set_xlabel('Data Source')
    ax_sources.set_ylabel('Number of Profiles')
    ax_sources.set_title('Profiles by Source')
    ax_sources.set_xticks(range(len(source_counts)))
    ax_sources.set_xticklabels(list(source_counts.keys()), rotation=45)
    
    # 4. Feature value distribution sample
    ax_features = fig.add_subplot(gs[1, 1:])
    # Sample some feature columns for distribution
    feature_cols = [col for col in collected_profiles.columns if col.startswith('X_')]
    sample_features = np.random.choice(feature_cols, min(5, len(feature_cols)), replace=False)
    
    for i, col in enumerate(sample_features):
        values = collected_profiles[col].to_numpy()
        ax_features.hist(values, bins=50, alpha=0.6, label=f'{col}', density=True)
    
    ax_features.set_xlabel('Feature Values')
    ax_features.set_ylabel('Density')
    ax_features.set_title('Sample Feature Distributions (Normalized)')
    ax_features.legend()
    ax_features.grid(True, alpha=0.3)
    
    return fig

def visualize_orf_sample_images(selected_orfs, num_samples=3, channels=["DNA", "RNA", "ER", "AGP", "Mito"]):
    """Visualize sample ORF images with all channels - similar to pilot exploration"""
    
    # Channel color mapping (similar to pilot)
    channel_colors = {
        "DNA": "blue",      # Hoechst - blue
        "RNA": "yellow",    # SYTO 14 - yellow 
        "ER": "green",      # Concanavalin A - green
        "AGP": "orange",    # Phalloidin/WGA - orange
        "Mito": "red"       # MitoTracker - red
    }
    
    fig, axes = plt.subplots(num_samples, len(channels) + 1, figsize=(24, 6*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    successfully_processed = 0
    
    for sample_idx in range(num_samples):
        if sample_idx >= len(selected_orfs):
            break
            
        orf_id = selected_orfs[sample_idx]
        print(f"Processing sample {sample_idx + 1}: {orf_id}")
        
        try:
            # Get image location info
            orf_info = get_item_location_info(orf_id)
            
            if orf_info.shape[0] == 0:
                print(f"  No images found for {orf_id}")
                continue
            
            # Get first available image
            test_row = orf_info.row(0)
            source = test_row[orf_info.columns.index("Metadata_Source")]
            batch = test_row[orf_info.columns.index("Metadata_Batch")]
            plate = test_row[orf_info.columns.index("Metadata_Plate")]
            well = test_row[orf_info.columns.index("Metadata_Well")]
            site = test_row[orf_info.columns.index("Metadata_Site")]
            
            print(f"  Location: {source}/{batch}/{plate}/{well}/site_{site}")
            
            # Load all channel images
            channel_images = {}
            all_imgs_norm = []
            
            for ch_idx, channel in enumerate(channels):
                try:
                    img = get_jump_image(source, batch, plate, well, channel, site, None)
                    channel_images[channel] = img
                    
                    # Normalize for display
                    img_norm = img.astype(float)
                    if img_norm.max() > img_norm.min():
                        img_norm = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min())
                    all_imgs_norm.append(img_norm)
                    
                    # Create custom colormap for this channel
                    color = channel_colors.get(channel, "gray")
                    cmap = mcolors.LinearSegmentedColormap.from_list(
                        f"custom_{color}", ["black", color]
                    )
                    
                    # Display individual channel
                    axes[sample_idx, ch_idx].imshow(img_norm, cmap=cmap)
                    axes[sample_idx, ch_idx].set_title(f"{channel}\n({img.shape})")
                    axes[sample_idx, ch_idx].axis('off')
                    
                except Exception as e:
                    print(f"  Error loading {channel}: {str(e)}")
                    axes[sample_idx, ch_idx].text(0.5, 0.5, f"Error\n{channel}", 
                                                ha='center', va='center', 
                                                transform=axes[sample_idx, ch_idx].transAxes)
                    axes[sample_idx, ch_idx].axis('off')
            
            # Create composite image (if we have all channels)
            if len(all_imgs_norm) == 5:
                composite = np.zeros((*all_imgs_norm[0].shape, 3))
                
                # Combine channels with appropriate color weighting
                # DNA (blue channel)
                composite[..., 2] = all_imgs_norm[0]
                # ER (green) + RNA (yellow-green contribution)  
                composite[..., 1] = 0.7*all_imgs_norm[2] + 0.3*all_imgs_norm[1]
                # Mito (red) + AGP (orange-red contribution)
                composite[..., 0] = 0.7*all_imgs_norm[4] + 0.3*all_imgs_norm[3]
                
                axes[sample_idx, -1].imshow(composite)
                axes[sample_idx, -1].set_title("5-Channel\nComposite")
                axes[sample_idx, -1].axis('off')
            
            # Add ORF label on the left
            axes[sample_idx, 0].text(-0.1, 0.5, f"ORF:\n{orf_id}", 
                                   rotation=90, ha='center', va='center', 
                                   transform=axes[sample_idx, 0].transAxes, fontsize=10)
            
            successfully_processed += 1
            
        except Exception as e:
            print(f"  Failed to process {orf_id}: {str(e)}")
            for col_idx in range(len(channels) + 1):
                axes[sample_idx, col_idx].text(0.5, 0.5, f"Error\n{orf_id}", 
                                             ha='center', va='center',
                                             transform=axes[sample_idx, col_idx].transAxes)
                axes[sample_idx, col_idx].axis('off')
    
    plt.suptitle(f'ORF Sample Images - {successfully_processed}/{num_samples} Successfully Loaded', 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    
    return fig

def visualize_orf_metadata_distributions(profiles, selected_orfs):
    """Create detailed metadata distribution plots"""
    
    collected_profiles = profiles.collect()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    # 1. Well distribution
    wells = collected_profiles['Metadata_Well'].to_list()
    well_counts = Counter(wells)
    
    # Create well plate heatmap
    ax = axes[0]
    well_matrix = np.zeros((16, 24))  # Standard 384-well plate
    
    for well, count in well_counts.items():
        if len(well) >= 3:
            row = ord(well[0]) - ord('A')  # A=0, B=1, etc.
            col = int(well[1:]) - 1  # 01=0, 02=1, etc.
            if 0 <= row < 16 and 0 <= col < 24:
                well_matrix[row, col] = count
    
    im = ax.imshow(well_matrix, cmap='YlOrRd', aspect='auto')
    ax.set_title('Profile Count by Well Position')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_xticks(range(0, 24, 4))
    ax.set_xticklabels(range(1, 25, 4))
    ax.set_yticks(range(16))
    ax.set_yticklabels([chr(ord('A') + i) for i in range(16)])
    plt.colorbar(im, ax=ax)
    
    # 2. Plate distribution
    ax = axes[1]
    plates = collected_profiles['Metadata_Plate'].to_list()
    plate_counts = Counter(plates)
    
    # Show top 20 plates
    top_plates = dict(plate_counts.most_common(20))
    ax.bar(range(len(top_plates)), list(top_plates.values()), color='skyblue')
    ax.set_title('Top 20 Plates by Profile Count')
    ax.set_xlabel('Plate (ranked)')
    ax.set_ylabel('Number of Profiles')
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    
    # 3. Source breakdown
    ax = axes[2]
    sources = collected_profiles['Metadata_Source'].to_list()
    source_counts = Counter(sources)
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(source_counts)))
    wedges, texts, autotexts = ax.pie(list(source_counts.values()), 
                                     labels=list(source_counts.keys()),
                                     autopct='%1.1f%%', colors=colors)
    ax.set_title('Profile Distribution by Source')
    
    # 4. Selected ORFs profile counts
    ax = axes[3]
    selected_counts = [Counter(collected_profiles['Metadata_JCP2022'].to_list())[orf] 
                      for orf in selected_orfs[:20]]  # First 20 selected ORFs
    
    ax.bar(range(len(selected_counts)), selected_counts, color='lightgreen')
    ax.set_title('Profile Counts for First 20 Selected ORFs')
    ax.set_xlabel('Selected ORF (index)')
    ax.set_ylabel('Number of Profiles')
    ax.axhline(y=5, color='red', linestyle='--', alpha=0.7, label='Expected (5)')
    ax.legend()
    
    # 5. Feature correlation matrix (sample)
    ax = axes[4]
    feature_cols = [col for col in collected_profiles.columns if col.startswith('X_')]
    sample_features = np.random.choice(feature_cols, min(10, len(feature_cols)), replace=False)
    
    corr_data = collected_profiles.select(sample_features).to_numpy()
    corr_matrix = np.corrcoef(corr_data.T)
    
    im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title('Feature Correlation Matrix (Sample)')
    ax.set_xticks(range(len(sample_features)))
    ax.set_yticks(range(len(sample_features)))
    ax.set_xticklabels([f'X_{i}' for i in range(len(sample_features))], rotation=45)
    ax.set_yticklabels([f'X_{i}' for i in range(len(sample_features))])
    plt.colorbar(im, ax=ax)
    
    # 6. Dataset summary comparison
    ax = axes[5]
    ax.axis('off')
    
    # Calculate some key statistics
    total_orfs = len(collected_profiles['Metadata_JCP2022'].unique())
    modal_profiles = 5  # We know this from earlier analysis
    selected_count = len(selected_orfs)
    
    summary_text = f"""
    Dataset Selection Summary
    ═══════════════════════
    
    Total Available ORFs:     {total_orfs:,}
    Modal Profiles per ORF:   {modal_profiles}
    
    Selected for Dataset:     {selected_count}
    Expected Total Images:    {selected_count * modal_profiles:,}
    
    Coverage:
    • {selected_count/total_orfs*100:.1f}% of available ORFs
    • Standardized to {modal_profiles} profiles each
    • Focus on high-quality, consistent data
    
    Next Steps:
    ✓ Validate image retrieval
    ✓ Create combined channel images
    ✓ Generate dataset metadata
    """
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, 
            fontsize=11, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    return fig

# Main execution function
def run_orf_dataset_visualization(profiles, selected_orfs):
    """Run all ORF dataset visualizations"""
    
    print("=== Creating ORF Dataset Visualizations ===")
    
    # 1. Dataset overview
    print("1. Generating dataset overview...")
    fig1 = visualize_orf_dataset_overview(profiles)
    plt.show()
    
    # 2. Sample images
    print("2. Loading sample ORF images...")
    fig2 = visualize_orf_sample_images(selected_orfs, num_samples=3)
    plt.show()
    
    # 3. Metadata distributions  
    print("3. Analyzing metadata distributions...")
    fig3 = visualize_orf_metadata_distributions(profiles, selected_orfs)
    plt.show()
    
    print("=== ORF Dataset Visualization Complete ===")
    
    return fig1, fig2, fig3

# Example usage:
# fig1, fig2, fig3 = run_orf_dataset_visualization(profiles, selected_orfs)

In [16]:
# ORF Gene Name Mapping Solution
print("=== Loading ORF Metadata for Gene Name Mapping ===")

import requests
import pandas as pd
from io import StringIO
import gzip

def load_orf_metadata_table():
    """Load the ORF metadata table that contains JCP2022 -> Gene Symbol mapping"""
    
    # Based on the search results, the ORF metadata should be available
    possible_urls = [
        "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.csv",
        "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.csv.gz", 
        "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.tsv",
        "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.tsv.gz"
    ]
    
    for url in possible_urls:
        try:
            print(f"Trying: {url}")
            response = requests.get(url, timeout=10)
            
            if response.status_code == 200:
                print(f"✓ Successfully loaded ORF metadata from: {url}")
                
                # Handle different file formats
                if url.endswith('.gz'):
                    content = gzip.decompress(response.content).decode('utf-8')
                else:
                    content = response.text
                    
                # Determine separator
                if '.tsv' in url:
                    separator = '\t'
                else:
                    separator = ','
                    
                # Parse the data
                df = pd.read_csv(StringIO(content), sep=separator)
                
                print(f"ORF metadata shape: {df.shape}")
                print(f"Columns: {list(df.columns)}")
                
                # Check for the key columns we need
                required_columns = ['Metadata_JCP2022']
                gene_symbol_columns = [col for col in df.columns if 'Gene' in col or 'Symbol' in col]
                
                print(f"Gene-related columns found: {gene_symbol_columns}")
                
                if len(gene_symbol_columns) > 0:
                    return df, gene_symbol_columns[0]  # Return df and gene column name
                else:
                    print("⚠ No gene symbol column found in this file")
                    return df, None
                    
            else:
                print(f"✗ Status {response.status_code}")
                
        except Exception as e:
            print(f"✗ Failed to load {url}: {str(e)}")
            continue
    
    return None, None

# Load ORF metadata
orf_metadata_df, gene_column = load_orf_metadata_table()

def create_jcp_to_gene_mapping(orf_metadata_df, gene_column):
    """Create a mapping dictionary from JCP2022 IDs to gene symbols"""
    
    if orf_metadata_df is None or gene_column is None:
        print("Cannot create mapping - ORF metadata not available")
        return {}
    
    # Filter out rows with missing gene symbols
    valid_mapping = orf_metadata_df.dropna(subset=[gene_column, 'Metadata_JCP2022'])
    
    # Create the mapping dictionary
    jcp_to_gene = dict(zip(valid_mapping['Metadata_JCP2022'], valid_mapping[gene_column]))
    
    print(f"Created mapping for {len(jcp_to_gene)} JCP2022 IDs to gene symbols")
    
    # Show some examples
    sample_mappings = list(jcp_to_gene.items())[:10]
    print("Sample mappings:")
    for jcp_id, gene_symbol in sample_mappings:
        print(f"  {jcp_id} -> {gene_symbol}")
    
    return jcp_to_gene

# Create the mapping
jcp_to_gene_mapping = create_jcp_to_gene_mapping(orf_metadata_df, gene_column)

def test_gene_symbol_image_access(jcp_ids, jcp_to_gene_mapping):
    """Test image access using gene symbols derived from JCP IDs"""
    
    print(f"\n=== Testing Gene Symbol Image Access ===")
    
    successful_mappings = []
    failed_mappings = []
    
    for jcp_id in jcp_ids[:5]:  # Test first 5
        if jcp_id in jcp_to_gene_mapping:
            gene_symbol = jcp_to_gene_mapping[jcp_id]
            print(f"\nTesting {jcp_id} -> {gene_symbol}")
            
            try:
                # Try get_item_location_info with gene symbol
                gene_info = get_item_location_info(gene_symbol)
                print(f"  ✓ SUCCESS: Found {gene_info.shape[0]} images for {gene_symbol}")
                successful_mappings.append((jcp_id, gene_symbol, gene_info))
                
            except Exception as e:
                print(f"  ✗ FAILED: {str(e)}")
                failed_mappings.append((jcp_id, gene_symbol, str(e)))
        else:
            print(f"\n✗ No gene mapping found for {jcp_id}")
            failed_mappings.append((jcp_id, "N/A", "No mapping"))
    
    print(f"\n=== Results Summary ===")
    print(f"Successful: {len(successful_mappings)}")
    print(f"Failed: {len(failed_mappings)}")
    
    return successful_mappings, failed_mappings

# Test with your selected ORFs if mapping exists
if jcp_to_gene_mapping and len(jcp_to_gene_mapping) > 0:
    successful_mappings, failed_mappings = test_gene_symbol_image_access(
        selected_orfs, jcp_to_gene_mapping
    )
    
    if successful_mappings:
        print(f"\n🎯 SOLUTION FOUND!")
        print(f"ORF images can be accessed using gene symbols")
        print(f"Next step: Update dataset creation to use gene symbols")
    else:
        print(f"\n⚠ Gene symbols found but image access still failing")
        print(f"May need additional investigation")
else:
    print(f"\n⚠ Could not load ORF metadata mapping")
    print(f"Will need to investigate alternative approaches")

# Alternative approach: Try broad_babel if metadata loading fails
def try_broad_babel_jcp_mapping():
    """Try broad_babel for JCP to gene mapping as fallback"""
    print(f"\n=== Trying broad_babel as Alternative ===")
    
    try:
        from broad_babel.query import get_mapper
        
        # Try to get a mapper that can convert JCP2022 to gene symbols
        mapper_configs = [
            ('standard_key', 'JCP2022', 'gene_symbol'),
            ('JCP2022', 'standard_key', 'gene_symbol'),
            ('pert_iname', 'JCP2022', 'gene_symbol'),
        ]
        
        for query_type, input_col, output_col in mapper_configs:
            try:
                print(f"Trying mapper: {query_type} {input_col} -> {output_col}")
                
                mapper = get_mapper(
                    query=query_type,
                    input_column=input_col, 
                    output_columns=[output_col]
                )
                
                # Test with first ORF
                test_result = mapper.query(selected_orfs[0])
                if test_result and output_col in test_result:
                    gene_symbol = test_result[output_col]
                    print(f"  ✓ SUCCESS: {selected_orfs[0]} -> {gene_symbol}")
                    
                    # Test image access
                    try:
                        gene_info = get_item_location_info(gene_symbol)
                        print(f"  ✓ Image access SUCCESS: {gene_info.shape[0]} images")
                        return mapper, output_col
                    except Exception as img_error:
                        print(f"  ✗ Image access failed: {str(img_error)}")
                else:
                    print(f"  ✗ No result returned")
                    
            except Exception as mapper_error:
                print(f"  ✗ Mapper failed: {str(mapper_error)}")
                continue
                
        return None, None
        
    except Exception as e:
        print(f"broad_babel approach failed: {str(e)}")
        return None, None

# Try broad_babel if direct metadata loading failed
if not successful_mappings and len(jcp_to_gene_mapping) == 0:
    babel_mapper, babel_gene_col = try_broad_babel_jcp_mapping()
    
    if babel_mapper:
        print(f"✓ broad_babel mapping successful!")
        print(f"Can proceed with dataset creation using broad_babel")
    else:
        print(f"✗ Both metadata table and broad_babel approaches failed")

print(f"\n=== Next Steps ===")
if successful_mappings or (jcp_to_gene_mapping and len(jcp_to_gene_mapping) > 0):
    print(f"1. ✓ JCP2022 -> Gene Symbol mapping available")
    print(f"2. ✓ Image access working via gene symbols")
    print(f"3. → Update dataset creation function to use gene mapping")
    print(f"4. → Create ORF dataset with actual images")
elif babel_mapper:
    print(f"1. ✓ broad_babel mapping available as fallback")
    print(f"2. → Implement dataset creation with broad_babel mapping")
else:
    print(f"1. ⚠ Need to investigate ORF metadata access further")
    print(f"2. → Contact JUMP consortium for guidance")
    print(f"3. → Continue with placeholder dataset for pipeline testing")

=== Loading ORF Metadata for Gene Name Mapping ===
Trying: https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.csv
✗ Status 404
Trying: https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.csv.gz
✓ Successfully loaded ORF metadata from: https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/orf.csv.gz
ORF metadata shape: (15132, 12)
Columns: ['Metadata_JCP2022', 'Metadata_broad_sample', 'Metadata_Name', 'Metadata_Vector', 'Metadata_Transcript', 'Metadata_Symbol', 'Metadata_NCBI_Gene_ID', 'Metadata_Taxon_ID', 'Metadata_Gene_Description', 'Metadata_Prot_Match', 'Metadata_Insert_Length', 'Metadata_pert_type']
Gene-related columns found: ['Metadata_Symbol', 'Metadata_NCBI_Gene_ID', 'Metadata_Gene_Description']
Created mapping for 15097 JCP2022 IDs to gene symbols
Sample mappings:
  JCP2022_900002 -> NAT1
  JCP2022_900003 -> AANAT
  JCP2022_900004 -> ABAT
  JCP2022_900005 -> ACADVL
  JCP2022_900006 -> ASIC1
  JCP2022_9

worker #3: 100%|██████████| 3/3 [00:04<00:00,  1.35s/it]


  ✓ SUCCESS: Found 108 images for GPAT4

Testing JCP2022_906196 -> NCF2


worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.26s/it]


  ✓ SUCCESS: Found 90 images for NCF2

Testing JCP2022_907074 -> MPHOSPH6


...

worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.36s/it]


  ✓ SUCCESS: Found 99 images for FAAP24

Testing JCP2022_914858 -> PRPF39


worker #4:  50%|█████     | 1/2 [00:01<00:01,  1.39s/it]

  ✓ SUCCESS: Found 90 images for PRPF39

=== Results Summary ===
Successful: 5
Failed: 0

🎯 SOLUTION FOUND!
ORF images can be accessed using gene symbols
Next step: Update dataset creation to use gene symbols

=== Next Steps ===
1. ✓ JCP2022 -> Gene Symbol mapping available
2. ✓ Image access working via gene symbols
3. → Update dataset creation function to use gene mapping
4. → Create ORF dataset with actual images


worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.40s/it]


In [17]:
# Final ORF μSplit Dataset Creation with Gene Symbol Mapping
print("=== Creating Complete ORF μSplit Dataset ===")

def prepare_orf_microsplit_dataset_final(
    selected_orfs,
    jcp_to_gene_mapping,
    profiles,
    output_dir="orf_microsplit_dataset",
    images_per_orf=5,
    channels_to_combine=["DNA", "RNA", "ER", "AGP", "Mito"],
    weights=None,
    normalize=False,
    seed=42
):
    """
    Complete ORF dataset creation using gene symbol mapping
    
    Parameters
    ----------
    selected_orfs: list
        List of JCP2022 IDs to process
    jcp_to_gene_mapping: dict
        Dictionary mapping JCP2022 IDs to gene symbols
    profiles: pl.LazyFrame
        ORF profiles dataframe
    output_dir: str
        Output directory path
    images_per_orf: int
        Number of images per ORF
    channels_to_combine: list
        Channels to combine
    weights: list or None
        Channel weights (None for equal weighting)
    normalize: bool
        Whether to normalize images
    seed: int
        Random seed
        
    Returns
    -------
    dict
        Dataset creation results
    """
    
    import os
    import numpy as np
    import tifffile
    import pandas as pd
    
    print(f"Creating ORF dataset: {len(selected_orfs)} ORFs × {images_per_orf} images = {len(selected_orfs) * images_per_orf} total images")
    
    # Create directory structure
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "combined"), exist_ok=True)
    for channel in channels_to_combine:
        os.makedirs(os.path.join(output_dir, channel), exist_ok=True)
    
    # Initialize tracking variables
    dataset_info = {
        "combined_images": [],
        "original_images": {channel: [] for channel in channels_to_combine},
        "metadata": [],
        "successful_orfs": [],
        "failed_orfs": []
    }
    
    np.random.seed(seed)
    image_counter = 0
    
    # Process each ORF
    for orf_idx, jcp_id in enumerate(selected_orfs):
        print(f"\nProcessing ORF {orf_idx+1}/{len(selected_orfs)}: {jcp_id}")
        
        # Check if we have gene mapping
        if jcp_id not in jcp_to_gene_mapping:
            print(f"  ✗ No gene mapping found for {jcp_id}")
            dataset_info["failed_orfs"].append(jcp_id)
            continue
            
        gene_symbol = jcp_to_gene_mapping[jcp_id]
        print(f"  Gene symbol: {gene_symbol}")
        
        try:
            # Get image location info using gene symbol
            gene_info = get_item_location_info(gene_symbol)
            print(f"  Found {gene_info.shape[0]} total images")
            
            # Filter for ORF plate type
            orf_samples = gene_info.filter(pl.col("Metadata_PlateType") == "ORF")
            
            if orf_samples.shape[0] < images_per_orf:
                print(f"  ⚠ Only {orf_samples.shape[0]} ORF images available, skipping")
                dataset_info["failed_orfs"].append(jcp_id)
                continue
            
            # Randomly select images
            indices = np.random.choice(
                orf_samples.shape[0],
                size=images_per_orf,
                replace=False
            )
            
            orf_success_count = 0
            
            # Process each selected image
            for i in indices:
                try:
                    # Extract image location metadata
                    row = orf_samples.row(i)
                    source = row[gene_info.columns.index("Metadata_Source")]
                    batch = row[gene_info.columns.index("Metadata_Batch")]
                    plate = row[gene_info.columns.index("Metadata_Plate")]
                    well = row[gene_info.columns.index("Metadata_Well")]
                    site = row[gene_info.columns.index("Metadata_Site")]
                    
                    print(f"    Processing image {orf_success_count+1}: {source}/{batch}/{plate}/{well}/site_{site}")
                    
                    # Retrieve all channel images
                    channel_images = {}
                    channel_success = True
                    
                    for channel in channels_to_combine:
                        try:
                            img = get_jump_image(source, batch, plate, well, channel, site, None)
                            channel_images[channel] = img
                        except Exception as channel_error:
                            print(f"      ✗ {channel} failed: {str(channel_error)}")
                            channel_success = False
                            break
                    
                    if not channel_success:
                        continue
                    
                    # Combine channels using the reference function
                    combined_img, stats = combine_channels_for_microsplit(
                        channel_images,
                        channels_to_combine=channels_to_combine,
                        weights=weights,
                        normalize=normalize
                    )
                    
                    # Save combined image with standardized naming
                    combined_filename = f"img_{image_counter:05d}_combined.tif"
                    combined_path = os.path.join(output_dir, "combined", combined_filename)
                    tifffile.imwrite(combined_path, combined_img)
                    dataset_info["combined_images"].append(combined_path)
                    
                    # Save individual channel images
                    for channel in channels_to_combine:
                        channel_filename = f"img_{image_counter:05d}_{channel}.tif"
                        channel_path = os.path.join(output_dir, channel, channel_filename)
                        tifffile.imwrite(channel_path, channel_images[channel])
                        dataset_info["original_images"][channel].append(channel_path)
                    
                    # Create comprehensive metadata entry
                    metadata_entry = {
                        # Core identifiers
                        "image_id": image_counter,
                        "orf_perturbation": jcp_id,
                        "gene_symbol": gene_symbol,
                        "pert_type": "orf",
                        
                        # Image location metadata
                        "source": source,
                        "batch": batch,
                        "plate": plate,
                        "well": well,
                        "site": site,
                        
                        # Channel and processing info
                        "combined_channels": channels_to_combine,
                        "weights": weights if weights else [1.0/len(channels_to_combine)] * len(channels_to_combine),
                        "normalized": normalize,
                        
                        # Image statistics
                        "min_val": stats["min_val"],
                        "max_val": stats["max_val"],
                        "mean_val": stats["mean_val"],
                        "std_val": stats["std_val"],
                        
                        # Dataset info for pipeline compatibility
                        "dataset": "orf",
                        "experiment": "microsplit_orf",
                        "channel_combination": "_".join(channels_to_combine),
                        
                        # File paths
                        "combined_path": f"combined/{combined_filename}",
                        **{f"{channel.lower()}_path": f"{channel}/img_{image_counter:05d}_{channel}.tif" 
                           for channel in channels_to_combine}
                    }
                    
                    dataset_info["metadata"].append(metadata_entry)
                    image_counter += 1
                    orf_success_count += 1
                    
                except Exception as img_error:
                    print(f"      ✗ Image processing failed: {str(img_error)}")
                    continue
            
            if orf_success_count > 0:
                dataset_info["successful_orfs"].append(jcp_id)
                print(f"  ✓ Successfully processed {orf_success_count}/{images_per_orf} images from {jcp_id} ({gene_symbol})")
            else:
                dataset_info["failed_orfs"].append(jcp_id)
                print(f"  ✗ Failed to process any images from {jcp_id}")
                
        except Exception as orf_error:
            print(f"  ✗ ORF processing failed: {str(orf_error)}")
            dataset_info["failed_orfs"].append(jcp_id)
            continue
    
    # Save metadata as CSV
    if dataset_info["metadata"]:
        metadata_df = pd.DataFrame(dataset_info["metadata"])
        metadata_path = os.path.join(output_dir, "dataset_metadata.csv")
        metadata_df.to_csv(metadata_path, index=False)
        dataset_info["metadata_file"] = metadata_path
        
        print(f"\n=== Dataset Creation Complete ===")
        print(f"✓ Total images created: {image_counter}")
        print(f"✓ Successful ORFs: {len(dataset_info['successful_orfs'])}")
        print(f"✓ Failed ORFs: {len(dataset_info['failed_orfs'])}")
        print(f"✓ Success rate: {len(dataset_info['successful_orfs'])/len(selected_orfs)*100:.1f}%")
        print(f"✓ Metadata saved: {metadata_path}")
        print(f"✓ Combined images: {os.path.join(output_dir, 'combined')}")
        
        # Display metadata summary
        print(f"\nMetadata structure:")
        print(f"  Shape: {metadata_df.shape}")
        print(f"  Columns: {len(metadata_df.columns)}")
        print(f"  Compatible with 01_noisemodels: ✓")
        
    else:
        print(f"\n✗ No images were successfully processed")
    
    return dataset_info

# Execute the complete dataset creation
print("Ready to create complete ORF dataset!")
print("Execute the following when ready:")
print()
print("dataset_results = prepare_orf_microsplit_dataset_final(")
print("    selected_orfs=selected_orfs,")
print("    jcp_to_gene_mapping=jcp_to_gene_mapping,")
print("    profiles=profiles,")
print("    output_dir='orf_microsplit_dataset_final'")
print(")")

# Example execution (you can run this):
if len(jcp_to_gene_mapping) > 0 and len(selected_orfs) > 0:
    print(f"\n=== Executing Dataset Creation ===")
    
    # Create the final dataset 
    dataset_results = prepare_orf_microsplit_dataset_final(
        selected_orfs=selected_orfs[:10],  # Test with first 10 ORFs first
        jcp_to_gene_mapping=jcp_to_gene_mapping,
        profiles=profiles,
        output_dir='orf_microsplit_dataset_final',
        images_per_orf=5,
        seed=42
    )
    
    print(f"\n🎉 ORF μSplit Dataset Creation Complete!")
    print(f"Directory: orf_microsplit_dataset_final")
    print(f"Ready for 01_noisemodels notebook!")
    
else:
    print("Please ensure jcp_to_gene_mapping and selected_orfs are available first")

worker #3:   0%|          | 0/3 [00:00<?, ?it/s]

=== Creating Complete ORF μSplit Dataset ===
Ready to create complete ORF dataset!
Execute the following when ready:

dataset_results = prepare_orf_microsplit_dataset_final(
    selected_orfs=selected_orfs,
    jcp_to_gene_mapping=jcp_to_gene_mapping,
    profiles=profiles,
    output_dir='orf_microsplit_dataset_final'
)

=== Executing Dataset Creation ===
Creating ORF dataset: 10 ORFs × 5 images = 50 total images

Processing ORF 1/10: JCP2022_904597
  Gene symbol: GPAT4


worker #3: 100%|██████████| 3/3 [00:03<00:00,  1.31s/it]


  Found 108 total images
    Processing image 1: source_4/2021_05_17_Batch4/BR00123791/M09/site_9
    Processing image 2: source_4/2021_05_17_Batch4/BR00123790/M09/site_2
    Processing image 3: source_4/2021_05_17_Batch4/BR00123790/M09/site_9
    Processing image 4: source_4/2021_05_17_Batch4/BR00123786/M09/site_5
    Processing image 5: source_4/2021_05_17_Batch4/BR00123786/M09/site_7
  ✓ Successfully processed 5/5 images from JCP2022_904597 (GPAT4)

Processing ORF 2/10: JCP2022_906196
  Gene symbol: NCF2


worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.28s/it]


...

worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.23s/it]


  Found 99 total images
    Processing image 1: source_4/2021_08_30_Batch13/BR00123539/D16/site_5
    Processing image 2: source_4/2021_08_23_Batch12/BR00126708/D16/site_1
    Processing image 3: source_4/2021_07_12_Batch8/BR00125635/D16/site_8
    Processing image 4: source_4/2021_08_23_Batch12/BR00126708/D16/site_8
    Processing image 5: source_4/2021_06_07_Batch5/BR00123947/A19/site_8
  ✓ Successfully processed 5/5 images from JCP2022_915023 (FAAP24)

Processing ORF 5/10: JCP2022_914858
  Gene symbol: PRPF39


worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]


  Found 90 total images
    Processing image 1: source_4/2021_07_26_Batch9/BR00126045/N20/site_8
    Processing image 2: source_4/2021_06_14_Batch6/BR00124776/N15/site_1
    Processing image 3: source_4/2021_07_26_Batch9/BR00126045/N20/site_2
    Processing image 4: source_4/2021_06_14_Batch6/BR00124770/N15/site_5


KeyboardInterrupt: 

## Final step = ORF dataset creation (300 images)

In [23]:
# Final ORF μSplit Dataset Creation with Gene Symbol Mapping
print("=== Creating Complete ORF μSplit Dataset ===")

def prepare_orf_microsplit_dataset_final(
    selected_orfs,
    jcp_to_gene_mapping,
    profiles,
    output_dir="orf_microsplit_dataset",
    images_per_orf=5,
    channels_to_combine=["DNA", "RNA", "ER", "AGP", "Mito"],
    weights=None,
    normalize=False,
    seed=42
):
    """
    Complete ORF dataset creation using gene symbol mapping
    
    Parameters
    ----------
    selected_orfs: list
        List of JCP2022 IDs to process
    jcp_to_gene_mapping: dict
        Dictionary mapping JCP2022 IDs to gene symbols
    profiles: pl.LazyFrame
        ORF profiles dataframe
    output_dir: str
        Output directory path
    images_per_orf: int
        Number of images per ORF
    channels_to_combine: list
        Channels to combine
    weights: list or None
        Channel weights (None for equal weighting)
    normalize: bool
        Whether to normalize images
    seed: int
        Random seed
        
    Returns
    -------
    dict
        Dataset creation results
    """
    
    import os
    import numpy as np
    import tifffile
    import pandas as pd
    
    print(f"Creating ORF dataset: {len(selected_orfs)} ORFs × {images_per_orf} images = {len(selected_orfs) * images_per_orf} total images")
    
    # Create directory structure
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "combined"), exist_ok=True)
    for channel in channels_to_combine:
        os.makedirs(os.path.join(output_dir, channel), exist_ok=True)
    
    # Initialize tracking variables
    dataset_info = {
        "combined_images": [],
        "original_images": {channel: [] for channel in channels_to_combine},
        "metadata": [],
        "successful_orfs": [],
        "failed_orfs": []
    }
    
    np.random.seed(seed)
    image_counter = 0
    
    # Process each ORF
    for orf_idx, jcp_id in enumerate(selected_orfs):
        print(f"\nProcessing ORF {orf_idx+1}/{len(selected_orfs)}: {jcp_id}")
        
        # Check if we have gene mapping
        if jcp_id not in jcp_to_gene_mapping:
            print(f"  ✗ No gene mapping found for {jcp_id}")
            dataset_info["failed_orfs"].append(jcp_id)
            continue
            
        gene_symbol = jcp_to_gene_mapping[jcp_id]
        print(f"  Gene symbol: {gene_symbol}")
        
        try:
            # Get image location info using gene symbol
            gene_info = get_item_location_info(gene_symbol)
            print(f"  Found {gene_info.shape[0]} total images")
            
            # Filter for ORF plate type
            orf_samples = gene_info.filter(pl.col("Metadata_PlateType") == "ORF")
            
            if orf_samples.shape[0] < images_per_orf:
                print(f"  ⚠ Only {orf_samples.shape[0]} ORF images available, skipping")
                dataset_info["failed_orfs"].append(jcp_id)
                continue
            
            # Randomly select images
            indices = np.random.choice(
                orf_samples.shape[0],
                size=images_per_orf,
                replace=False
            )
            
            orf_success_count = 0
            
            # Process each selected image
            for i in indices:
                try:
                    # Extract image location metadata
                    row = orf_samples.row(i)
                    source = row[gene_info.columns.index("Metadata_Source")]
                    batch = row[gene_info.columns.index("Metadata_Batch")]
                    plate = row[gene_info.columns.index("Metadata_Plate")]
                    well = row[gene_info.columns.index("Metadata_Well")]
                    site = row[gene_info.columns.index("Metadata_Site")]
                    
                    print(f"    Processing image {orf_success_count+1}: {source}/{batch}/{plate}/{well}/site_{site}")
                    
                    # Retrieve all channel images
                    channel_images = {}
                    channel_success = True
                    
                    for channel in channels_to_combine:
                        try:
                            img = get_jump_image(source, batch, plate, well, channel, site, None)
                            channel_images[channel] = img
                        except Exception as channel_error:
                            print(f"      ✗ {channel} failed: {str(channel_error)}")
                            channel_success = False
                            break
                    
                    if not channel_success:
                        continue
                    
                    # Combine channels using the reference function
                    combined_img, stats = combine_channels_for_microsplit(
                        channel_images,
                        channels_to_combine=channels_to_combine,
                        weights=weights,
                        normalize=normalize
                    )
                    
                    # Save combined image with standardized naming
                    combined_filename = f"img_{image_counter:05d}_combined.tif"
                    combined_path = os.path.join(output_dir, "combined", combined_filename)
                    tifffile.imwrite(combined_path, combined_img)
                    dataset_info["combined_images"].append(combined_path)
                    
                    # Save individual channel images
                    for channel in channels_to_combine:
                        channel_filename = f"img_{image_counter:05d}_{channel}.tif"
                        channel_path = os.path.join(output_dir, channel, channel_filename)
                        tifffile.imwrite(channel_path, channel_images[channel])
                        dataset_info["original_images"][channel].append(channel_path)
                    
                    # Create comprehensive metadata entry
                    metadata_entry = {
                        # Core identifiers
                        "image_id": image_counter,
                        "orf_perturbation": jcp_id,
                        "gene_symbol": gene_symbol,
                        "pert_type": "orf",
                        
                        # Image location metadata
                        "source": source,
                        "batch": batch,
                        "plate": plate,
                        "well": well,
                        "site": site,
                        
                        # Channel and processing info
                        "combined_channels": channels_to_combine,
                        "weights": weights if weights else [1.0/len(channels_to_combine)] * len(channels_to_combine),
                        "normalized": normalize,
                        
                        # Image statistics
                        "min_val": stats["min_val"],
                        "max_val": stats["max_val"],
                        "mean_val": stats["mean_val"],
                        "std_val": stats["std_val"],
                        
                        # Dataset info for pipeline compatibility
                        "dataset": "orf",
                        "experiment": "microsplit_orf",
                        "channel_combination": "_".join(channels_to_combine),
                        
                        # File paths
                        "combined_path": f"combined/{combined_filename}",
                        **{f"{channel.lower()}_path": f"{channel}/img_{image_counter:05d}_{channel}.tif" 
                           for channel in channels_to_combine}
                    }
                    
                    dataset_info["metadata"].append(metadata_entry)
                    image_counter += 1
                    orf_success_count += 1
                    
                except Exception as img_error:
                    print(f"      ✗ Image processing failed: {str(img_error)}")
                    continue
            
            if orf_success_count > 0:
                dataset_info["successful_orfs"].append(jcp_id)
                print(f"  ✓ Successfully processed {orf_success_count}/{images_per_orf} images from {jcp_id} ({gene_symbol})")
            else:
                dataset_info["failed_orfs"].append(jcp_id)
                print(f"  ✗ Failed to process any images from {jcp_id}")
                
        except Exception as orf_error:
            print(f"  ✗ ORF processing failed: {str(orf_error)}")
            dataset_info["failed_orfs"].append(jcp_id)
            continue
    
    # Save metadata as CSV
    if dataset_info["metadata"]:
        metadata_df = pd.DataFrame(dataset_info["metadata"])
        metadata_path = os.path.join(output_dir, "dataset_metadata.csv")
        metadata_df.to_csv(metadata_path, index=False)
        dataset_info["metadata_file"] = metadata_path
        
        print(f"\n=== Dataset Creation Complete ===")
        print(f"✓ Total images created: {image_counter}")
        print(f"✓ Successful ORFs: {len(dataset_info['successful_orfs'])}")
        print(f"✓ Failed ORFs: {len(dataset_info['failed_orfs'])}")
        print(f"✓ Success rate: {len(dataset_info['successful_orfs'])/len(selected_orfs)*100:.1f}%")
        print(f"✓ Metadata saved: {metadata_path}")
        print(f"✓ Combined images: {os.path.join(output_dir, 'combined')}")
        
        # Display metadata summary
        print(f"\nMetadata structure:")
        print(f"  Shape: {metadata_df.shape}")
        print(f"  Columns: {len(metadata_df.columns)}")
        print(f"  Compatible with 01_noisemodels: ✓")
        
    else:
        print(f"\n✗ No images were successfully processed")
    
    return dataset_info

# Execute the complete dataset creation
print("Ready to create complete ORF dataset!")
print("Execute the following when ready:")
print()
print("dataset_results = prepare_orf_microsplit_dataset_final(")
print("    selected_orfs=selected_orfs,")
print("    jcp_to_gene_mapping=jcp_to_gene_mapping,")
print("    profiles=profiles,")
print("    output_dir='orf_microsplit_dataset_300images'")
print(")")

# Example execution (you can run this):
if len(jcp_to_gene_mapping) > 0 and len(selected_orfs) > 0:
    print(f"\n=== Executing Dataset Creation ===")
    
    # Select 50 ORFs randomly for 100-image dataset
    selected_orfs_100 = np.random.choice(selected_orfs, 50, replace=False).tolist()
    
    # Create the final dataset 
    dataset_results = prepare_orf_microsplit_dataset_final(
        selected_orfs=selected_orfs_100,
        jcp_to_gene_mapping=jcp_to_gene_mapping,
        profiles=profiles,
        output_dir='orf_rand6_100images',
        images_per_orf=2,  
        seed=777
    )
    
    print(f"\n🎉 ORF μSplit Dataset Creation Complete!")
    print(f"Directory: orf_biorand1_100images")
    print(f"Ready for 01_noisemodels notebook!")
    
else:
    print("Please ensure jcp_to_gene_mapping and selected_orfs are available first")

=== Creating Complete ORF μSplit Dataset ===
Ready to create complete ORF dataset!
Execute the following when ready:

dataset_results = prepare_orf_microsplit_dataset_final(
    selected_orfs=selected_orfs,
    jcp_to_gene_mapping=jcp_to_gene_mapping,
    profiles=profiles,
    output_dir='orf_microsplit_dataset_300images'
)

=== Executing Dataset Creation ===
Creating ORF dataset: 50 ORFs × 2 images = 100 total images

Processing ORF 1/50: JCP2022_904597
  Gene symbol: GPAT4


worker #2: 100%|██████████| 3/3 [00:04<00:00,  1.37s/it]


  Found 108 total images
    Processing image 1: source_4/2021_05_17_Batch4/BR00123786/M09/site_2
    Processing image 2: source_4/2021_05_17_Batch4/BR00123787/M09/site_2
  ✓ Successfully processed 2/2 images from JCP2022_904597 (GPAT4)

Processing ORF 2/50: JCP2022_901522
  Gene symbol: TAC3


worker #3: 100%|██████████| 3/3 [00:13<00:00,  4.38s/it]


  Found 108 total images
    Processing image 1: source_4/2021_06_21_Batch7/BR00124785/D06/site_1
    Processing image 2: source_4/2021_06_21_Batch7/BR00124784/D06/site_2
  ✓ Successfully processed 2/2 images from JCP2022_901522 (TAC3)

Processing ORF 3/50: JCP2022_910366
  Gene symbol: CCL3


...

  Found 45 total images
    Processing image 1: source_4/2021_07_12_Batch8/BR00125619/B12/site_3
    Processing image 2: source_4/2021_07_12_Batch8/BR00124787/B12/site_4
  ✓ Successfully processed 2/2 images from JCP2022_911314 (MCUB)

Processing ORF 10/50: JCP2022_912045
  Gene symbol: TRIM4


worker #4: 100%|██████████| 2/2 [00:03<00:00,  1.53s/it]


  Found 90 total images
    Processing image 1: source_4/2021_08_02_Batch10/BR00126398/N03/site_8
    Processing image 2: source_4/2021_08_02_Batch10/BR00126398/N03/site_3
  ✓ Successfully processed 2/2 images from JCP2022_912045 (TRIM4)

Processing ORF 11/50: JCP2022_905889
  Gene symbol: GNB2


worker #4: 100%|██████████| 2/2 [00:02<00:00,  1.30s/it]


  Found 90 total images
    Processing image 1: source_4/2021_05_10_Batch3/BR00123614/L23/site_4
    Processing image 2: source_4/2021_05_31_Batch2/BR00121543/P05/site_6
